# Adaptive Science Tutor — Reinforcement Learning Agentic System
### Hrishikesh Kulkarni | Take-Home Final Exam

This notebook implements a **hybrid RL-powered adaptive tutoring agent** that selects science questions based on each student's real-time knowledge, fatigue, and performance using three reinforcement learning methods.

---

## System Architecture

```
STUDENT STATE  (7-dimensional)
  ├─ topic_idx        [0-9]   Which topic the question covers
  ├─ difficulty       [0-2]   Easy / Medium / Hard
  ├─ acc_bucket       [0-2]   Recent accuracy: struggling / ok / good
  ├─ knowledge_bucket [0-2]   Topic knowledge: low / mid / high
  ├─ fatigue_bucket   [0-2]   Fresh / moderate / tired
  ├─ session_progress [0-2]   Early / mid / late session
  └─ repeat_flag      [0-1]   Has this question been asked before?
           │
           ├──► UCB Bandit ─────► best (topic, difficulty) zone
           │         │
           ├──► Q-Learning ─────► best specific question in zone
           │
           └──► DQN (7→64→64→500) ► ensemble re-score
                      │
                FINAL ACTION (0–499)
                      │
              Student answers (A/B/C/D)
                      │
                REWARD SIGNAL
                      │
         ┌────────────┴────────────┐
    Q-Table update          DQN replay buffer
    (Bellman eq.)           → mini-batch backprop
```

---

## How to Run
**Runtime → Run all** — training takes ~2 minutes, then click the URL in Cell 7.

## Cell 1 — Install Dependencies

In [ ]:
!pip install flask numpy matplotlib -q
print("Ready")

## Cell 2 — Question Bank (500 Questions, 10 Topics)

500 multiple-choice science questions across 10 disciplines, each with:
- A correct answer
- 3 carefully chosen wrong distractors
- A difficulty level (easy / medium / hard)
- An optional hint

| Topic | Questions |
|-------|-----------|
| Physics | 50 |
| Chemistry | 50 |
| Biology | 50 |
| Astronomy | 50 |
| Earth Science | 50 |
| Mathematics | 50 |
| Computer Science | 50 |
| History of Science | 50 |
| Environmental Science | 50 |
| Human Body | 50 |

In [ ]:
import subprocess, time, threading, random, os
subprocess.run(['pip', 'install', 'flask', '-q'], capture_output=True)
from flask import Flask, request, jsonify, Response
import numpy as np
from collections import deque, defaultdict
import pickle, json as _json

os.makedirs('/content/results', exist_ok=True)

# ═══════════════════════════════════════════════════════════════
# QUESTION BANK  (500 questions with A/B/C/D choices)
# ═══════════════════════════════════════════════════════════════
"""
questions_mc.py  –  500 questions with correct answers + 3 wrong distractors each
"""

TOPICS = [
    "physics", "chemistry", "biology", "astronomy", "earth_science",
    "mathematics", "computer_science", "history_of_science",
    "environmental_science", "human_body",
]

DIFFICULTY_NAMES = {0: "easy", 1: "medium", 2: "hard"}

def _q(qid, text, topic, diff, correct, wrong1, wrong2, wrong3, hint=""):
    return {
        "id": qid, "text": text, "topic": topic, "difficulty": diff,
        "correct": correct,
        "choices": [correct, wrong1, wrong2, wrong3],
        "hint": hint,
    }

QUESTIONS = [
    # ── PHYSICS (0–49) ──────────────────────────────────────────────────────
    _q(0,  "What is the SI unit of force?",                                   "physics",0,"Newton","Joule","Pascal","Watt","Named after a famous scientist"),
    _q(1,  "What particle has a negative electric charge?",                   "physics",0,"Electron","Proton","Neutron","Photon"),
    _q(2,  "What is Newton's first law also called?",                         "physics",0,"Law of Inertia","Law of Acceleration","Law of Action-Reaction","Law of Gravity"),
    _q(3,  "Which colour of visible light has the longest wavelength?",       "physics",0,"Red","Violet","Blue","Green"),
    _q(4,  "What does LED stand for?",                                        "physics",0,"Light Emitting Diode","Low Energy Device","Luminous Electric Display","Light Energy Driver"),
    _q(5,  "What is the unit of electrical resistance?",                      "physics",0,"Ohm","Volt","Ampere","Watt"),
    _q(6,  "Sound cannot travel through which medium?",                       "physics",0,"Vacuum","Water","Air","Steel"),
    _q(7,  "What type of mirror is used in car rear-view mirrors?",           "physics",0,"Convex","Concave","Plane","Parabolic"),
    _q(8,  "What force pulls objects toward the Earth?",                      "physics",0,"Gravity","Magnetism","Friction","Tension"),
    _q(9,  "What is the approximate speed of light in a vacuum?",             "physics",0,"3×10⁸ m/s","3×10⁶ m/s","3×10¹⁰ m/s","3×10⁴ m/s"),
    _q(10, "What is the formula for kinetic energy?",                         "physics",1,"½mv²","mv","mgh","½mv"),
    _q(11, "What phenomenon causes a straw to appear bent in water?",         "physics",1,"Refraction","Reflection","Diffraction","Dispersion"),
    _q(12, "What is the unit of electric current?",                           "physics",1,"Ampere","Volt","Ohm","Watt"),
    _q(13, "What does Ohm's Law state?",                                      "physics",1,"V = IR","P = IV","F = ma","E = mc²"),
    _q(14, "What type of wave is sound?",                                     "physics",1,"Longitudinal","Transverse","Electromagnetic","Surface"),
    _q(15, "What does conservation of energy state?",                         "physics",1,"Energy cannot be created or destroyed","Energy always increases","Energy is always lost as heat","Energy equals mass times speed"),
    _q(16, "What is terminal velocity?",                                      "physics",1,"Constant speed when drag equals gravity","Maximum possible speed","Speed of sound","Escape velocity from Earth"),
    _q(17, "What is the main difference between mass and weight?",            "physics",1,"Mass is constant, weight depends on gravity","They are the same thing","Weight is constant, mass changes","Mass is measured in Newtons"),
    _q(18, "What is electromagnetic induction?",                              "physics",1,"Generating voltage by changing magnetic flux","Creating light from electricity","Converting heat to electricity","Storing charge in a capacitor"),
    _q(19, "What is the Doppler effect?",                                     "physics",1,"Change in observed frequency due to relative motion","Bending of light around objects","Interference of two waves","Reflection of sound off surfaces"),
    _q(20, "What does Schrödinger's equation describe?",                      "physics",2,"Evolution of quantum wave functions","Motion of classical particles","Electromagnetic field propagation","Thermal radiation from black bodies"),
    _q(21, "What does the Heisenberg Uncertainty Principle state?",           "physics",2,"Position and momentum cannot both be precisely known","Nothing can travel faster than light","Energy is quantised","Electrons orbit in fixed shells"),
    _q(22, "What is a Higgs boson?",                                          "physics",2,"Particle that gives other particles mass","Anti-matter equivalent of an electron","Carrier of the electromagnetic force","Component of an atomic nucleus"),
    _q(23, "What is special relativity's most famous equation?",              "physics",2,"E = mc²","F = ma","PV = nRT","E = hf"),
    _q(24, "What is quantum entanglement?",                                   "physics",2,"Correlated quantum states of two particles","Splitting of an atom","Wave-particle duality","Tunnelling through a barrier"),
    _q(25, "What is the key difference between nuclear fission and fusion?",  "physics",2,"Fission splits nuclei; fusion combines them","Fission releases no energy; fusion does","Fission uses hydrogen; fusion uses uranium","Fission is cold; fusion is hot"),
    _q(26, "What is a black hole's event horizon?",                           "physics",2,"Boundary beyond which nothing can escape","The black hole's surface","The region of maximum gravity","The outer atmosphere of a black hole"),
    _q(27, "What is the photoelectric effect?",                               "physics",2,"Emission of electrons when light hits a metal","Splitting light into a spectrum","Converting photons to heat","Reflection of light by metals"),
    _q(28, "What is superconductivity?",                                      "physics",2,"Zero electrical resistance below critical temperature","Perfect heat conduction","Infinite magnetic permeability","Frictionless electron flow at any temperature"),
    _q(29, "What does the Standard Model of particle physics describe?",      "physics",2,"Fundamental particles and forces (except gravity)","All four fundamental forces","The structure of atomic nuclei only","The motion of planets and stars"),
    _q(30, "What is the SI unit of power?",                                   "physics",0,"Watt","Joule","Newton","Ampere"),
    _q(31, "What is potential energy?",                                       "physics",0,"Stored energy due to position or state","Energy of motion","Energy released as heat","Energy carried by light"),
    _q(32, "What type of lens converges light rays to a point?",              "physics",0,"Convex","Concave","Plane","Diverging"),
    _q(33, "What unit measures frequency?",                                   "physics",0,"Hertz","Decibel","Metre","Second"),
    _q(34, "What is the formula for pressure?",                               "physics",1,"Force divided by area","Mass times acceleration","Force times distance","Energy divided by time"),
    _q(35, "What is centripetal force?",                                      "physics",1,"Force directed toward the centre of circular motion","Force pushing outward in circular motion","Gravitational force between two bodies","Frictional force on a spinning object"),
    _q(36, "What does Archimedes' principle state?",                          "physics",1,"Buoyant force equals weight of displaced fluid","Objects always float in water","Pressure increases with depth","Denser objects sink faster"),
    _q(37, "What is a semiconductor?",                                        "physics",1,"Material with conductivity between conductors and insulators","Perfect conductor of electricity","Material that blocks all current","A type of superconductor"),
    _q(38, "What is magnetic flux?",                                          "physics",2,"Measure of magnetic field through a surface","Strength of a magnetic field at a point","Rate of change of magnetic field","Current induced by a magnetic field"),
    _q(39, "What is plasma in physics?",                                      "physics",1,"Ionised gas — the fourth state of matter","Liquid metal","Superheated solid","A type of electromagnetic wave"),
    _q(40, "What does Newton's third law state?",                             "physics",0,"Every action has an equal and opposite reaction","Force equals mass times acceleration","Objects in motion stay in motion","Acceleration is proportional to net force"),
    _q(41, "What is the SI unit of energy?",                                  "physics",0,"Joule","Watt","Newton","Pascal"),
    _q(42, "What is total internal reflection?",                              "physics",1,"Light reflects entirely when hitting boundary at critical angle","Light bends when entering a denser medium","Light splits into colours through a prism","Light slows down in a vacuum"),
    _q(43, "What is the key difference between AC and DC current?",           "physics",1,"AC alternates direction; DC flows one way","AC is faster than DC","DC is more dangerous than AC","AC only works in batteries"),
    _q(44, "What is a photon?",                                               "physics",1,"Particle/quantum of light","Negatively charged particle","Neutral particle in nucleus","Anti-matter particle"),
    _q(45, "What is the weak nuclear force responsible for?",                 "physics",2,"Radioactive beta decay","Holding the nucleus together","Electromagnetic interactions","Gravitational attraction"),
    _q(46, "What does de Broglie's hypothesis state?",                        "physics",2,"All matter has wave-like properties","Light is purely a particle","Electrons orbit in fixed paths","Energy comes in discrete packets"),
    _q(47, "What does the Pauli exclusion principle state?",                  "physics",2,"No two fermions can occupy the same quantum state","All particles attract each other","Quantum states are always uncertain","Energy is always conserved in reactions"),
    _q(48, "What does a transformer do?",                                     "physics",1,"Changes AC voltage using electromagnetic induction","Converts AC to DC","Stores electrical energy","Amplifies electrical signals"),
    _q(49, "What does the Stefan-Boltzmann law relate?",                      "physics",2,"Thermal radiation to temperature (T⁴)","Speed of light to wavelength","Force to distance","Pressure to volume"),

    # ── CHEMISTRY (50–99) ───────────────────────────────────────────────────
    _q(50, "What is the chemical symbol for gold?",                           "chemistry",0,"Au","Go","Gd","Ag"),
    _q(51, "What is the most abundant gas in Earth's atmosphere?",            "chemistry",0,"Nitrogen","Oxygen","Carbon dioxide","Argon"),
    _q(52, "What is the pH of pure water?",                                   "chemistry",0,"7","0","14","5"),
    _q(53, "How many elements are on the periodic table?",                    "chemistry",0,"118","92","106","112"),
    _q(54, "What is the chemical formula for water?",                         "chemistry",0,"H₂O","HO","H₂O₂","OH"),
    _q(55, "What is an atom's nucleus made of?",                              "chemistry",0,"Protons and neutrons","Protons and electrons","Neutrons only","Electrons and neutrons"),
    _q(56, "What is the chemical symbol for sodium?",                         "chemistry",0,"Na","So","Sd","Nm"),
    _q(57, "What type of bond involves sharing electrons?",                   "chemistry",1,"Covalent","Ionic","Metallic","Hydrogen"),
    _q(58, "What is Avogadro's number approximately?",                        "chemistry",1,"6.02×10²³","6.02×10²⁰","6.02×10²⁶","6.02×10¹²"),
    _q(59, "What is oxidation in chemistry?",                                 "chemistry",1,"Loss of electrons","Gain of electrons","Gain of protons","Loss of neutrons"),
    _q(60, "What is the difference between an element and a compound?",       "chemistry",1,"Elements are pure substances; compounds contain two or more elements chemically bonded","Elements contain multiple atoms; compounds have one","Elements are liquid; compounds are solid","Compounds cannot be broken down; elements can"),
    _q(61, "What is a catalyst?",                                             "chemistry",1,"Substance that speeds up a reaction without being consumed","Substance produced in a reaction","Substance that slows reactions","Substance that changes pH"),
    _q(62, "What is the atomic number of carbon?",                            "chemistry",0,"6","8","12","4"),
    _q(63, "What is an isotope?",                                             "chemistry",1,"Atoms of same element with different neutron counts","Atoms with different proton counts","Charged atoms","Different elements with same mass"),
    _q(64, "What distinguishes ionic bonds from covalent bonds?",             "chemistry",1,"Ionic involves electron transfer; covalent involves sharing","Ionic is stronger than covalent","Covalent involves metals; ionic involves non-metals","Ionic bonds occur in gases only"),
    _q(65, "What is molarity?",                                               "chemistry",1,"Moles of solute per litre of solution","Mass of solute per litre","Moles of solvent per litre","Density of a solution"),
    _q(66, "What does the ideal gas law state?",                              "chemistry",2,"PV = nRT","PV = nR","P = nRT","PV = RT"),
    _q(67, "What is a redox reaction?",                                       "chemistry",2,"Reaction involving simultaneous oxidation and reduction","Reaction at high temperature only","Reaction producing only water","Reaction between two acids"),
    _q(68, "What is enthalpy?",                                               "chemistry",2,"Total heat content of a system at constant pressure","Total kinetic energy of particles","Rate of a chemical reaction","Measure of disorder in a system"),
    _q(69, "What does Le Chatelier's principle state?",                       "chemistry",2,"A system at equilibrium shifts to oppose disturbances","Reactions always favour the products","Temperature has no effect on equilibrium","Pressure only affects solids"),
    _q(70, "What is electronegativity?",                                      "chemistry",1,"Atom's ability to attract electrons in a bond","Atom's number of electrons","Atom's tendency to lose electrons","Strength of an ionic bond"),
    _q(71, "What is the octet rule?",                                         "chemistry",1,"Atoms tend to gain/lose electrons to achieve 8 valence electrons","All atoms have 8 electrons","Atoms share exactly 8 bonds","Molecules always have 8 atoms"),
    _q(72, "What is a buffer solution?",                                      "chemistry",2,"Solution that resists pH changes","Solution with pH of exactly 7","Solution with maximum acidity","Solution containing only water and salt"),
    _q(73, "What is chirality in chemistry?",                                 "chemistry",2,"Non-superimposable mirror image property of molecules","Molecule's ability to conduct electricity","Bond angle in a tetrahedral molecule","Symmetric arrangement of atoms"),
    _q(74, "What does Hess's Law state?",                                     "chemistry",2,"Total enthalpy change is path-independent","Reactions always release heat","Products have less energy than reactants","Enthalpy equals entropy times temperature"),
    _q(75, "What distinguishes organic from inorganic chemistry?",            "chemistry",1,"Organic studies carbon compounds; inorganic studies others","Organic is natural; inorganic is synthetic","Organic uses only gases; inorganic uses solids","Organic molecules are always larger"),
    _q(76, "What is a titration?",                                            "chemistry",1,"Technique to find concentration using a known-concentration solution","Process of heating a solution","Separating mixtures by boiling points","Measuring pH with an electrode"),
    _q(77, "What are noble gases?",                                           "chemistry",0,"Unreactive elements in Group 18","Highly reactive metals","Gases that combust easily","Elements with one valence electron"),
    _q(78, "What is the chemical formula for table salt?",                    "chemistry",0,"NaCl","KCl","NaF","CaCl₂"),
    _q(79, "What is sublimation?",                                            "chemistry",1,"Solid converting directly to gas","Gas converting to liquid","Liquid converting to solid","Solid melting into liquid"),
    _q(80, "What is a mole in chemistry?",                                    "chemistry",1,"6.02×10²³ particles of a substance","A unit of temperature","A unit of pressure","A measure of solution concentration"),
    _q(81, "What is activation energy?",                                      "chemistry",2,"Minimum energy required to start a reaction","Energy released during a reaction","Energy stored in chemical bonds","Difference in energy between reactants and products"),
    _q(82, "What is polymerisation?",                                         "chemistry",2,"Process of joining monomers into polymer chains","Breaking polymers into monomers","Separating a mixture into components","Forming ionic crystals from solution"),
    _q(83, "What is the key difference between acids and bases?",             "chemistry",0,"Acids donate H⁺ ions; bases accept them","Acids are red; bases are blue","Acids have pH above 7; bases below 7","Acids are solids; bases are liquids"),
    _q(84, "What is electrolysis?",                                           "chemistry",1,"Using electricity to drive a non-spontaneous chemical reaction","Spontaneous flow of electrons","Separation of liquids by boiling","Formation of crystals from solution"),
    _q(85, "What is a hydrogen bond?",                                        "chemistry",1,"Attraction between H bonded to electronegative atom and another electronegative atom","Covalent bond involving hydrogen","Ionic bond in water molecules","Bond between two hydrogen atoms"),
    _q(86, "What does the periodic law state?",                               "chemistry",1,"Properties of elements repeat periodically with atomic number","All elements have the same properties","Elements are arranged by mass only","Noble gases appear first in each period"),
    _q(87, "What are alkali metals?",                                         "chemistry",1,"Highly reactive metals in Group 1 of the periodic table","Metals that don't react with water","Metals in the centre of the periodic table","Non-reactive metals like gold and silver"),
    _q(88, "What distinguishes exothermic from endothermic reactions?",       "chemistry",1,"Exothermic releases heat; endothermic absorbs heat","Exothermic is faster; endothermic is slower","Exothermic produces gas; endothermic does not","Exothermic needs a catalyst; endothermic does not"),
    _q(89, "What does nuclear chemistry study?",                              "chemistry",2,"Changes in atomic nuclei and radioactivity","Reactions between noble gases","Organic compounds containing radioactive atoms","Chemical reactions at extremely high temperatures"),
    _q(90, "What is a coordination compound?",                                "chemistry",2,"Complex with central metal atom bonded to ligands","Compound formed only from metals","Molecule with equal numbers of all atoms","Polymer formed from metal ions"),
    _q(91, "What is thermodynamic entropy?",                                  "chemistry",2,"Measure of disorder or randomness in a system","Total energy of a system","Heat released by a reaction","Rate of energy transfer"),
    _q(92, "What is the Born-Haber cycle used for?",                          "chemistry",2,"Calculating lattice energy of ionic compounds","Finding activation energy of reactions","Predicting reaction rates","Determining bond polarity"),
    _q(93, "What is a precipitation reaction?",                               "chemistry",1,"Reaction forming an insoluble solid from two solutions","Reaction producing a gas","Reaction between acid and base","Reaction requiring electricity"),
    _q(94, "What is spectroscopy used for in chemistry?",                     "chemistry",2,"Identifying substances by their interaction with light","Measuring reaction rates","Separating chemical mixtures","Synthesising organic compounds"),
    _q(95, "What distinguishes a mixture from a compound?",                   "chemistry",0,"Mixtures are not chemically bonded; compounds are","Mixtures are always liquid; compounds are solid","Compounds can be separated physically; mixtures cannot","Mixtures have fixed compositions; compounds do not"),
    _q(96, "What role do enzymes play as biological catalysts?",              "chemistry",1,"Speed up biological reactions without being consumed","Provide energy for reactions","Act as reactants in metabolism","Store energy in cells"),
    _q(97, "What is distillation?",                                           "chemistry",0,"Separating liquids by differences in boiling points","Mixing liquids together","Filtering solid particles from a liquid","Evaporating a liquid completely"),
    _q(98, "What does a Lewis structure show?",                               "chemistry",1,"Valence electrons and bonding in a molecule","3D shape of a molecule","Energy levels of electrons","Nuclear structure of an atom"),
    _q(99, "What is stoichiometry?",                                          "chemistry",2,"Quantitative relationship between reactants and products","Study of reaction rates","Study of heat in reactions","Study of electrochemical reactions"),

    # ── BIOLOGY (100–149) ───────────────────────────────────────────────────
    _q(100,"What is the powerhouse of the cell?",                             "biology",0,"Mitochondria","Nucleus","Ribosome","Golgi apparatus"),
    _q(101,"How many chromosomes do humans normally have?",                   "biology",0,"46","23","48","44"),
    _q(102,"What molecule carries genetic information?",                      "biology",0,"DNA","RNA","ATP","Protein"),
    _q(103,"What process do plants use to make food from sunlight?",          "biology",0,"Photosynthesis","Respiration","Fermentation","Transpiration"),
    _q(104,"What is the basic unit of life?",                                 "biology",0,"Cell","Atom","Molecule","Organ"),
    _q(105,"What is the function of red blood cells?",                        "biology",0,"Carry oxygen around the body","Fight infection","Clot blood","Produce hormones"),
    _q(106,"What is natural selection?",                                      "biology",0,"Survival and reproduction of best-adapted organisms","Random changes in DNA","Process of artificial breeding","Inheritance of acquired characteristics"),
    _q(107,"What is the key difference between DNA and RNA?",                 "biology",1,"DNA is double-stranded with thymine; RNA is single-stranded with uracil","DNA is found in cytoplasm; RNA in nucleus","DNA carries amino acids; RNA carries genes","DNA is shorter than RNA"),
    _q(108,"What is mitosis?",                                                "biology",1,"Cell division producing two identical daughter cells","Cell division producing four sex cells","Process of DNA replication only","Programmed cell death"),
    _q(109,"What is an ecosystem?",                                           "biology",0,"Community of organisms and their physical environment","Only the living organisms in an area","The physical environment without living things","A single population of one species"),
    _q(110,"What is the role of ribosomes?",                                  "biology",1,"Synthesise proteins from mRNA","Package and modify proteins","Produce energy for the cell","Digest cellular waste"),
    _q(111,"What is meiosis?",                                                "biology",1,"Cell division producing four genetically unique sex cells","Cell division producing two identical cells","DNA replication process","Programmed cell death"),
    _q(112,"What is a dominant allele?",                                      "biology",1,"Allele whose trait is expressed even with one copy","Allele only expressed with two copies","Most common allele in a population","Allele on the X chromosome only"),
    _q(113,"What distinguishes prokaryotes from eukaryotes?",                 "biology",1,"Prokaryotes lack a membrane-bound nucleus; eukaryotes have one","Prokaryotes are larger than eukaryotes","Prokaryotes have mitochondria; eukaryotes do not","Prokaryotes are only found in animals"),
    _q(114,"What is the function of the cell membrane?",                      "biology",1,"Controls what enters and exits the cell","Produces energy for the cell","Contains genetic information","Synthesises proteins"),
    _q(115,"What is protein synthesis?",                                      "biology",2,"Process of making proteins from genetic code via transcription and translation","Direct copying of DNA","Breaking proteins into amino acids","Transport of proteins to the nucleus"),
    _q(116,"What is epigenetics?",                                            "biology",2,"Study of heritable changes in gene expression not involving DNA sequence changes","Study of mutations in DNA","Analysis of the entire genome","Inheritance of genetic diseases only"),
    _q(117,"What is the Hardy-Weinberg equilibrium?",                         "biology",2,"State where allele frequencies don't change across generations (no evolution)","Speed of genetic mutations","Rate of natural selection","Ratio of dominant to recessive alleles"),
    _q(118,"What is horizontal gene transfer?",                               "biology",2,"Transfer of genetic material between organisms not via reproduction","Normal passing of genes from parent to offspring","Mutation caused by radiation","Deletion of genes from a chromosome"),
    _q(119,"What is the CRISPR-Cas9 system?",                                 "biology",2,"Gene editing tool that can cut and modify DNA precisely","Method of DNA sequencing","Process of cloning organisms","Technique for growing cells in culture"),
    _q(120,"What are stem cells?",                                            "biology",1,"Undifferentiated cells that can develop into specialised cell types","Cells that produce hormones","Cells that fight infection","Cells that carry oxygen"),
    _q(121,"What is the main difference between mitosis and meiosis?",        "biology",1,"Mitosis produces 2 identical cells; meiosis produces 4 unique sex cells","Mitosis is for sex cells; meiosis is for body cells","Mitosis involves crossing over; meiosis does not","Mitosis reduces chromosome number; meiosis does not"),
    _q(122,"What is an enzyme?",                                              "biology",1,"Protein that acts as a biological catalyst","Sugar that provides energy","Lipid that forms cell membranes","Nucleic acid that carries genetic info"),
    _q(123,"What is homeostasis?",                                            "biology",1,"Maintenance of a stable internal environment","Growth of an organism","Response to external stimuli only","Reproduction of cells"),
    _q(124,"What is the role of ATP in cells?",                               "biology",1,"Primary energy currency of the cell","Carrier of genetic information","Building block of proteins","Component of cell membranes"),
    _q(125,"What is osmosis?",                                                "biology",1,"Movement of water across a semipermeable membrane from low to high solute concentration","Movement of all molecules from high to low concentration","Active transport of ions","Movement of water against concentration gradient"),
    _q(126,"What is a food chain?",                                           "biology",0,"Linear sequence showing how energy passes between organisms","Circular network of all organisms in an ecosystem","List of all species in an area","Relationship between predators only"),
    _q(127,"What is biodiversity?",                                           "biology",0,"Variety of life in an area (species, genes, ecosystems)","Number of individuals in a species","Total biomass in an ecosystem","Genetic similarity within a species"),
    _q(128,"What is the key difference between a virus and a bacterium?",     "biology",1,"Viruses need a host cell to reproduce; bacteria are independent living cells","Viruses are larger than bacteria","Bacteria cause more disease than viruses","Viruses have cell walls; bacteria do not"),
    _q(129,"What is evolution?",                                              "biology",0,"Change in heritable characteristics of a population over successive generations","Individual organisms changing during their lifetime","Deliberate selective breeding","Random genetic mutation only"),
    _q(130,"What is gene expression?",                                        "biology",2,"Process by which information in a gene produces a functional product (protein/RNA)","Visible physical traits of an organism","Mutation of a gene","Transfer of genes between species"),
    _q(131,"What is apoptosis?",                                              "biology",2,"Programmed cell death","Uncontrolled cell division","Cell growth in response to damage","Process of cell differentiation"),
    _q(132,"What is the central dogma of molecular biology?",                 "biology",2,"DNA → RNA → Protein","Protein → RNA → DNA","RNA → DNA → Protein","DNA → Protein → RNA"),
    _q(133,"What is symbiosis?",                                              "biology",1,"Close long-term interaction between two different species","Competition between species for resources","Predator-prey relationship only","Seasonal migration of animals"),
    _q(134,"What are the four main tissue types in animals?",                 "biology",1,"Epithelial, connective, muscle, nervous","Bone, blood, muscle, skin","Cardiac, smooth, skeletal, neural","Endocrine, exocrine, vascular, lymphatic"),
    _q(135,"What distinguishes aerobic from anaerobic respiration?",          "biology",1,"Aerobic uses oxygen and produces more ATP; anaerobic does not use oxygen","Aerobic occurs in animals; anaerobic in plants","Aerobic produces lactic acid; anaerobic produces CO₂","Aerobic is faster; anaerobic is slower"),
    _q(136,"What is a transgenic organism?",                                  "biology",2,"Organism containing DNA from another species","Organism produced by cloning","Organism that has undergone natural mutation","Organism with extra chromosomes"),
    _q(137,"What is the endosymbiotic theory?",                               "biology",2,"Mitochondria and chloroplasts originated as free-living bacteria absorbed by cells","All cells evolved from a single virus","Eukaryotes gave rise to prokaryotes","Cell organelles arose from nuclear folding"),
    _q(138,"What is a species?",                                              "biology",0,"Group of organisms that can interbreed and produce fertile offspring","All organisms in the same geographical area","All organisms with similar appearance","Group of organisms sharing the same diet"),
    _q(139,"What distinguishes autotrophs from heterotrophs?",                "biology",1,"Autotrophs make their own food; heterotrophs consume other organisms","Autotrophs are animals; heterotrophs are plants","Autotrophs are larger than heterotrophs","Autotrophs require oxygen; heterotrophs do not"),
    _q(140,"What is genetic drift?",                                          "biology",2,"Random change in allele frequencies in small populations","Natural selection acting on a population","Deliberate change in genes by scientists","Migration of individuals between populations"),
    _q(141,"What is the role of chlorophyll?",                                "biology",1,"Absorbs light energy for photosynthesis","Stores glucose in plant cells","Transports water through plants","Breaks down toxins in cells"),
    _q(142,"What is a plasmid?",                                              "biology",2,"Small circular DNA molecule in bacteria, separate from chromosomal DNA","Linear strand of RNA in plant cells","Protective coat around a virus","Segment of chromosomal DNA in eukaryotes"),
    _q(143,"What is a trophic level?",                                        "biology",1,"Position of an organism in a food chain","Number of species in an ecosystem","Total energy in an ecosystem","Geographic area occupied by a species"),
    _q(144,"What does the Golgi apparatus do?",                               "biology",1,"Modifies, packages and ships proteins","Produces energy for the cell","Synthesises proteins","Contains the cell's DNA"),
    _q(145,"What is a mutation?",                                             "biology",1,"Change in the DNA sequence","Transfer of genes between organisms","Programmed cell death","Process of protein synthesis"),
    _q(146,"What is the role of the lysosome?",                               "biology",1,"Digest cellular waste and foreign material using enzymes","Produce energy for the cell","Synthesise lipids","Package and export proteins"),
    _q(147,"What is a reflex arc?",                                           "biology",1,"Neural pathway allowing automatic responses to stimuli without brain involvement","Path of blood through the heart","Sequence of muscle contractions","Pathway of hormones through the bloodstream"),
    _q(148,"What is a phylogenetic tree?",                                    "biology",2,"Diagram showing evolutionary relationships among species","Map of an organism's habitat","Diagram of cell division stages","Chart of an organism's growth over time"),
    _q(149,"What distinguishes innate from adaptive immunity?",               "biology",2,"Innate is non-specific and immediate; adaptive is specific and develops over time","Innate involves antibodies; adaptive does not","Innate is slower than adaptive","Innate only occurs in vertebrates"),

    # ── ASTRONOMY (150–199) ─────────────────────────────────────────────────
    _q(150,"What is the closest planet to the Sun?",                          "astronomy",0,"Mercury","Venus","Mars","Earth"),
    _q(151,"What is the largest planet in our solar system?",                 "astronomy",0,"Jupiter","Saturn","Neptune","Uranus"),
    _q(152,"What is a light-year?",                                           "astronomy",0,"Distance light travels in one year","Time light takes to reach Earth","Speed of light per second","Brightness of a star"),
    _q(153,"What is the name of our galaxy?",                                 "astronomy",0,"Milky Way","Andromeda","Triangulum","Whirlpool"),
    _q(154,"How many planets are in our solar system?",                       "astronomy",0,"8","9","7","10"),
    _q(155,"What is the Sun primarily composed of?",                          "astronomy",0,"Hydrogen and helium","Oxygen and nitrogen","Iron and nickel","Carbon and oxygen"),
    _q(156,"What is a meteor shower?",                                        "astronomy",0,"Earth passing through debris trail left by a comet","Stars exploding in the sky","Comets entering the solar system","Asteroids colliding in the asteroid belt"),
    _q(157,"What causes seasons on Earth?",                                   "astronomy",1,"Earth's axial tilt as it orbits the Sun","Earth's varying distance from the Sun","Changes in the Sun's energy output","The Moon's gravitational influence"),
    _q(158,"What is a nebula?",                                               "astronomy",1,"Cloud of gas and dust in space where stars can form","A dying star","A type of galaxy","A cluster of black holes"),
    _q(159,"What is the difference between a solar and lunar eclipse?",       "astronomy",1,"Solar: Moon blocks Sun; Lunar: Earth's shadow falls on Moon","Solar: Earth blocks Moon; Lunar: Moon blocks Sun","Solar occurs at night; Lunar occurs during day","Solar lasts longer than lunar"),
    _q(160,"What is a neutron star?",                                         "astronomy",2,"Ultra-dense remnant of a massive star after supernova collapse","A star with no nuclear reactions","A star made entirely of dark matter","A young star in early formation"),
    _q(161,"What is the Big Bang theory?",                                    "astronomy",1,"Universe originated from an extremely hot, dense point ~13.8 billion years ago","Universe has always existed and is unchanging","Universe was created by a massive explosion from outside","Stars formed before the universe expanded"),
    _q(162,"What is dark matter?",                                            "astronomy",2,"Undetected matter accounting for ~27% of universe detected only by gravity","Matter in the dark side of a planet","Black holes that emit no light","Antimatter in interstellar space"),
    _q(163,"What is a pulsar?",                                               "astronomy",2,"Rapidly rotating neutron star emitting regular beams of radiation","A type of quasar near a black hole","A star that pulses in brightness","An exploding star"),
    _q(164,"What is the Chandrasekhar limit?",                                "astronomy",2,"Maximum mass (~1.4 solar masses) of a stable white dwarf","Minimum mass needed for a star to form","Distance at which a planet can have liquid water","Speed at which a star collapses"),
    _q(165,"What distinguishes a planet from a dwarf planet?",                "astronomy",1,"Planets clear their orbital neighbourhood; dwarf planets do not","Planets orbit the Sun; dwarf planets orbit other planets","Planets have moons; dwarf planets do not","Planets are larger than stars"),
    _q(166,"What is a comet?",                                                "astronomy",0,"Icy body that develops a tail of gas and dust when near the Sun","Rocky body in the asteroid belt","A type of small moon","Fragment of a destroyed planet"),
    _q(167,"What does the Hubble constant measure?",                          "astronomy",2,"Rate of expansion of the universe","Distance to the nearest galaxy","Speed of light in a vacuum","Brightness of the most distant stars"),
    _q(168,"What is gravitational lensing?",                                  "astronomy",2,"Bending of light by massive objects' gravity","Magnification of stars by telescopes","Refraction of light through a planet's atmosphere","Reflection of light off asteroid surfaces"),
    _q(169,"What is the cosmic microwave background?",                        "astronomy",2,"Thermal radiation leftover from early universe (~380,000 years after Big Bang)","Light from the most distant galaxies","Radiation emitted by black holes","Microwave signals from pulsars"),
    _q(170,"What is a supernova?",                                            "astronomy",1,"Powerful explosion at the end of a massive star's life","A new star being born","A collision between two galaxies","A star increasing in brightness periodically"),
    _q(171,"What shape is the Milky Way galaxy?",                             "astronomy",0,"Spiral","Elliptical","Irregular","Spherical"),
    _q(172,"What is the asteroid belt?",                                      "astronomy",0,"Region between Mars and Jupiter containing many asteroids","Ring of dust around Saturn","Belt of comets beyond Neptune","Region of dark matter around the galaxy"),
    _q(173,"What is the Oort Cloud?",                                         "astronomy",2,"Distant spherical shell of icy bodies surrounding the solar system","Dense region of asteroids beyond Pluto","Cloud of gas surrounding the Sun","Ring system around Neptune"),
    _q(174,"What is stellar nucleosynthesis?",                                "astronomy",2,"Creation of heavier elements from lighter ones inside stars","Process by which stars form from nebulae","Splitting of atoms inside a star","Fusion of galaxies over time"),
    _q(175,"What is the habitable zone around a star?",                       "astronomy",1,"Region where liquid water could exist on a planet's surface","Zone where no radiation exists","Area closest to a star","Region beyond a star's influence"),
    _q(176,"What is the difference between rotation and revolution of a planet?","astronomy",1,"Rotation: spinning on its axis; Revolution: orbiting the Sun","Rotation: orbiting the Sun; Revolution: spinning on its axis","Both terms mean the same thing","Rotation applies to moons; revolution to planets"),
    _q(177,"What is a binary star system?",                                   "astronomy",1,"Two stars orbiting a common centre of mass","A star with exactly two planets","A double galaxy","A star that appears twice as bright as normal"),
    _q(178,"What is redshift used to measure?",                               "astronomy",2,"Recession velocity and distance of galaxies (universe expansion)","Temperature of a star","Composition of a planet's atmosphere","Rotation speed of a black hole"),
    _q(179,"What is the approximate escape velocity of Earth?",               "astronomy",2,"11.2 km/s","7.9 km/s","3.0 km/s","30 km/s"),
    _q(180,"What are sunspots?",                                              "astronomy",1,"Cooler, darker regions on the Sun caused by magnetic activity","Hot spots on the Sun's surface","Storms in the Sun's upper atmosphere","Regions where solar wind originates"),
    _q(181,"What causes the aurora borealis?",                                "astronomy",1,"Solar wind particles interacting with Earth's magnetic field and atmosphere","Reflection of sunlight off ice at the poles","Lightning storms in the upper atmosphere","Volcanic eruptions releasing gas"),
    _q(182,"What is Jupiter's Great Red Spot?",                               "astronomy",1,"Massive storm larger than Earth that has lasted hundreds of years","A volcanic region on Jupiter's surface","A large impact crater","A concentration of dark matter"),
    _q(183,"What distinguishes reflecting from refracting telescopes?",       "astronomy",1,"Reflecting uses mirrors; refracting uses lenses","Reflecting uses lenses; refracting uses mirrors","Reflecting is smaller; refracting is larger","Reflecting is for radio waves; refracting for visible light"),
    _q(184,"What is the main sequence of stars?",                             "astronomy",2,"Phase where stars fuse hydrogen to helium in their cores","Sequence of star formation from nebula","Order of brightness among stars","Lifecycle stages of a dying star"),
    _q(185,"What is a Hertzsprung-Russell diagram?",                          "astronomy",2,"Plot of stellar luminosity vs temperature showing star types and lifecycle","Map of star positions in the sky","Chart of galaxy distances","Diagram of the solar system"),
    _q(186,"What is tidal locking?",                                          "astronomy",2,"When rotation period equals orbital period (same face always faces partner)","When a moon escapes a planet's gravity","When tides stop due to distance","When two bodies orbit each other at equal speed"),
    _q(187,"What is an exoplanet?",                                           "astronomy",1,"Planet orbiting a star outside our solar system","A planet without a moon","A rogue planet with no star","A dwarf planet in our solar system"),
    _q(188,"What is the Kuiper Belt?",                                        "astronomy",2,"Region beyond Neptune containing icy bodies including Pluto","Belt of asteroids between Mars and Jupiter","Cloud of comets beyond the Oort Cloud","Ring of dark matter around the galaxy"),
    _q(189,"What is perihelion?",                                             "astronomy",1,"Point in orbit when a body is closest to the Sun","Point when a body is farthest from the Sun","Centre of a planet's orbit","Time when Earth crosses the Sun's equatorial plane"),
    _q(190,"What causes the phases of the Moon?",                             "astronomy",0,"Different portions of the Moon's lit face visible from Earth as it orbits","Moon moving closer and farther from Earth","Earth's shadow falling on the Moon","Changes in the Moon's reflected light"),
    _q(191,"What is solar wind?",                                             "astronomy",1,"Stream of charged particles (plasma) ejected from the Sun's upper atmosphere","Light radiation from the Sun","Magnetic field lines from the Sun","Gravitational waves from solar flares"),
    _q(192,"What did the geocentric model incorrectly claim?",                "astronomy",0,"Earth is at the centre of the universe","Earth orbits the Sun","Stars are other suns","The Moon causes tides"),
    _q(193,"What is an accretion disk?",                                      "astronomy",2,"Disk of gas and dust spiralling into a central massive object","Ring system around a planet","Flat region of an elliptical galaxy","Dust cloud surrounding a nebula"),
    _q(194,"What is the Fermi paradox?",                                      "astronomy",2,"Contradiction between probability of extraterrestrial life and lack of evidence","Impossibility of faster-than-light travel","Paradox about time travel near black holes","Contradiction in measurements of the Hubble constant"),
    _q(195,"What distinguishes a meteor from a meteorite?",                   "astronomy",1,"A meteor burns up in atmosphere; a meteorite reaches the ground","A meteor is larger than a meteorite","A meteorite is in space; a meteor is on the ground","They are the same object"),
    _q(196,"What is the Roche limit?",                                        "astronomy",2,"Distance within which tidal forces break apart a satellite","Maximum distance a moon can orbit a planet","Boundary of a star's gravitational influence","Distance at which planets can be detected"),
    _q(197,"What is a quasar?",                                               "astronomy",2,"Extremely luminous active galactic nucleus powered by a supermassive black hole","A type of neutron star","A large cluster of galaxies","A very bright supernova remnant"),
    _q(198,"What is stellar parallax?",                                       "astronomy",2,"Apparent shift of a star's position as Earth orbits the Sun, used to measure distance","Brightness variation of a star","Spectrum shift of a star's light","Wobble of a star caused by orbiting planets"),
    _q(199,"What is the cosmic distance ladder?",                             "astronomy",2,"Series of methods used to measure increasingly large astronomical distances","Sequence of star formation stages","Hierarchy of galaxy sizes","Classification of stars by luminosity"),

    # ── EARTH SCIENCE (200–249) ─────────────────────────────────────────────
    _q(200,"What are the three types of rocks?",                              "earth_science",0,"Igneous, sedimentary, metamorphic","Granite, limestone, marble","Hard, soft, medium","Volcanic, oceanic, continental"),
    _q(201,"What causes earthquakes?",                                        "earth_science",0,"Movement of tectonic plates","Volcanic eruptions only","Underground water flow","Changes in atmospheric pressure"),
    _q(202,"What is the water cycle?",                                        "earth_science",0,"Continuous movement of water through evaporation, condensation, and precipitation","Flow of water in rivers only","Process of water purification","Movement of groundwater"),
    _q(203,"What is the inner core of the Earth primarily made of?",          "earth_science",1,"Solid iron and nickel","Liquid magma","Silicon and oxygen","Granite and basalt"),
    _q(204,"What is tectonic plate movement?",                                "earth_science",1,"Motion of large sections of Earth's crust driven by mantle convection","Shifting of continents due to ocean tides","Movement caused by Earth's magnetic field","Slow erosion of continental edges"),
    _q(205,"What distinguishes weather from climate?",                        "earth_science",0,"Weather is short-term atmospheric conditions; climate is long-term patterns","Weather is warmer than climate","Climate changes daily; weather does not","Weather occurs globally; climate is local"),
    _q(206,"What is an igneous rock?",                                        "earth_science",0,"Rock formed from cooled magma or lava","Rock formed from compressed sediments","Rock changed by heat and pressure","Rock formed at the ocean floor only"),
    _q(207,"What does the Richter scale measure?",                            "earth_science",1,"Magnitude (energy released) of an earthquake","Depth of an earthquake","Duration of an earthquake","Speed of seismic waves"),
    _q(208,"What is erosion?",                                                "earth_science",0,"Wearing away and transport of rock/soil by wind, water, or ice","Deposition of sediment in water","Splitting of rocks by temperature change","Volcanic ash covering the ground"),
    _q(209,"What is the greenhouse effect?",                                  "earth_science",0,"Trapping of heat by atmospheric gases, warming Earth's surface","Cooling of Earth due to cloud cover","Reflection of sunlight by ice caps","Absorption of UV radiation by the ozone layer"),
    _q(210,"What is the difference between magma and lava?",                  "earth_science",0,"Magma is molten rock underground; lava is magma that has reached the surface","Lava is hotter than magma","Magma is on the surface; lava is underground","They are the same substance"),
    _q(211,"What is a fossil?",                                               "earth_science",0,"Preserved remains or traces of ancient organisms in rock","Any old rock formation","Imprint of a living organism on soil","Crystal formed from ancient minerals"),
    _q(212,"What is the rock cycle?",                                         "earth_science",1,"Continuous transformation of rocks between igneous, sedimentary, and metamorphic types","Weathering of rocks over time only","Formation of rocks from magma only","Cycle of volcanic eruptions"),
    _q(213,"What was Pangaea?",                                               "earth_science",1,"Supercontinent that existed ~300 million years ago before breaking apart","First ocean on Earth","Ancient mountain range","Name for Earth's first atmosphere"),
    _q(214,"What distinguishes a hurricane from a tornado?",                  "earth_science",1,"Hurricanes form over oceans and are much larger; tornadoes form over land and are smaller","Hurricanes are faster than tornadoes","Tornadoes form over water; hurricanes over land","Hurricanes have higher wind speeds than tornadoes"),
    _q(215,"What is soil primarily composed of?",                             "earth_science",0,"Minerals, organic matter, water, and air","Only sand and clay","Pure minerals and rock","Water and dissolved minerals"),
    _q(216,"What is the carbon cycle?",                                       "earth_science",1,"Movement of carbon through atmosphere, oceans, organisms, and Earth's crust","Burning of fossil fuels only","Process of photosynthesis only","Exchange of carbon between plants and animals"),
    _q(217,"What is a seismograph used for?",                                 "earth_science",0,"Detecting and recording seismic waves from earthquakes","Measuring volcanic gas emissions","Predicting tsunamis","Measuring ocean depth"),
    _q(218,"What is the Mohorovičić discontinuity?",                          "earth_science",2,"Boundary between Earth's crust and mantle","Boundary between mantle and outer core","Deepest point in the ocean","Boundary between inner and outer core"),
    _q(219,"What is isostasy?",                                               "earth_science",2,"Gravitational equilibrium between Earth's crust and mantle","Equal distribution of tectonic plates","Constant sea level worldwide","Balance between volcanic and erosional forces"),
    _q(220,"What causes the tides?",                                          "earth_science",1,"Gravitational pull of the Moon (and Sun) on Earth's oceans","Earth's rotation only","Wind patterns across the oceans","Changes in ocean temperature"),
    _q(221,"What is subduction?",                                             "earth_science",2,"Process where one tectonic plate slides beneath another into the mantle","Splitting of tectonic plates apart","Collision of two continental plates","Sideways sliding of plates past each other"),
    _q(222,"What is radiometric dating?",                                     "earth_science",2,"Using radioactive decay rates to determine the age of rocks and fossils","Dating rocks by their colour and texture","Measuring rock hardness to estimate age","Using magnetic patterns to date ocean floor"),
    _q(223,"What distinguishes convergent from divergent plate boundaries?",  "earth_science",2,"Convergent: plates collide; Divergent: plates move apart","Convergent: plates slide past; Divergent: plates collide","Convergent forms oceans; Divergent forms mountains","They are the same type of boundary"),
    _q(224,"What is the hydrosphere?",                                        "earth_science",1,"All water on, in, and above the Earth","Only the oceans","Water vapour in the atmosphere only","Underground water only"),
    _q(225,"What is the stratosphere?",                                       "earth_science",1,"Layer of atmosphere above troposphere containing the ozone layer","Lowest layer of atmosphere where weather occurs","Outermost layer of atmosphere","Layer between mesosphere and thermosphere"),
    _q(226,"What is a glacier?",                                              "earth_science",0,"Large slow-moving mass of ice formed from compacted snow","Frozen lake","Ice formation on a mountain peak","Sea ice in polar regions"),
    _q(227,"What is the nitrogen cycle?",                                     "earth_science",1,"Movement of nitrogen through atmosphere, soil, and living organisms","Production of nitrogen in the atmosphere","Conversion of nitrogen to oxygen by plants","Loss of nitrogen from the atmosphere"),
    _q(228,"What is metamorphic rock?",                                       "earth_science",1,"Rock transformed by heat and pressure without melting","Rock formed from cooled lava","Rock formed from compressed sediments","Rock formed on the ocean floor"),
    _q(229,"What distinguishes stalactites from stalagmites?",                "earth_science",0,"Stalactites hang from ceiling; stalagmites rise from floor","Stalactites are on the floor; stalagmites on the ceiling","They are the same formation","Stalactites are made of granite; stalagmites of limestone"),
    _q(230,"What is liquefaction in geoscience?",                             "earth_science",2,"Saturated soil behaving like liquid during an earthquake","Melting of rock deep underground","Conversion of lava to rock","Erosion of soil by water"),
    _q(231,"What is the El Niño phenomenon?",                                 "earth_science",2,"Periodic warming of central and eastern Pacific Ocean affecting global weather","Cold current in the Pacific Ocean","Seasonal wind pattern in the Atlantic","Volcanic eruption cycle in the Pacific"),
    _q(232,"What distinguishes an ocean from a sea?",                         "earth_science",0,"Oceans are larger and not fully enclosed by land; seas are smaller and partially enclosed","Seas are saltier than oceans","Oceans are shallower than seas","Seas are in the southern hemisphere only"),
    _q(233,"What is a mineral?",                                              "earth_science",0,"Naturally occurring, inorganic crystalline solid with definite chemical composition","Any rock found in the ground","Organic material found in soil","Liquid metal found in Earth's core"),
    _q(234,"What is permafrost?",                                             "earth_science",1,"Ground that remains frozen for at least two consecutive years","Seasonal snow cover","Ice shelf floating in the ocean","Frozen volcanic rock"),
    _q(235,"What commonly causes a tsunami?",                                 "earth_science",1,"Underwater earthquake, volcanic eruption, or landslide displacing water","Strong ocean winds","Tidal forces from the Moon","Rapid change in ocean temperature"),
    _q(236,"What is a biome?",                                                "earth_science",0,"Large region characterised by its climate, plants, and animals","Single ecosystem in a small area","Ocean zone at a specific depth","Climate zone defined only by temperature"),
    _q(237,"What is the thermocline?",                                        "earth_science",2,"Layer in the ocean where temperature drops rapidly with increasing depth","Surface layer of warm water","Deep ocean layer with constant temperature","Zone where cold and warm ocean currents meet"),
    _q(238,"What is albedo?",                                                 "earth_science",2,"Fraction of incoming solar radiation reflected by a surface","Total solar energy absorbed by Earth","Rate of heat loss from Earth's surface","Measure of atmospheric transparency"),
    _q(239,"What is geothermal energy?",                                      "earth_science",1,"Heat energy from Earth's interior used to generate electricity","Energy from the Sun stored in rocks","Energy released by volcanic eruptions only","Chemical energy stored in fossils"),
    _q(240,"What distinguishes longitude from latitude?",                     "earth_science",0,"Latitude measures north-south; longitude measures east-west","Longitude measures north-south; latitude east-west","Latitude measures distance from the equator in km","Longitude is measured in metres"),
    _q(241,"What is a hotspot volcano?",                                      "earth_science",2,"Volcano formed by a stationary plume of hot mantle material, not at a plate boundary","Volcano at a convergent boundary","Volcano caused by continental collision","Volcano at a mid-ocean ridge"),
    _q(242,"What is the lithosphere?",                                        "earth_science",1,"Rigid outer layer of Earth comprising the crust and upper mantle","Only the Earth's crust","The entire mantle","The solid inner core"),
    _q(243,"What is the Coriolis effect?",                                    "earth_science",2,"Deflection of moving objects due to Earth's rotation (right in NH, left in SH)","Slowing of wind due to surface friction","Rising of warm air near the equator","Circular motion of ocean currents due to tides"),
    _q(244,"What is groundwater?",                                            "earth_science",0,"Water stored in underground aquifers in rock and soil","Water flowing in rivers","Water vapour in the atmosphere","Water frozen in glaciers"),
    _q(245,"What is sedimentary rock primarily formed from?",                 "earth_science",1,"Compacted layers of sediment (sand, silt, organic material)","Cooled magma","Rock changed by pressure and heat","Crystallised minerals from solution"),
    _q(246,"What distinguishes a valley from a canyon?",                      "earth_science",0,"Canyons are deeper and have steeper walls; valleys are broader with gentler slopes","They are the same landform","Valleys are deeper than canyons","Canyons are formed by glaciers; valleys by rivers"),
    _q(247,"What is the asthenosphere?",                                      "earth_science",2,"Semi-molten, ductile layer of upper mantle on which tectonic plates move","Solid layer directly beneath Earth's crust","Outer layer of Earth's core","Region of highest pressure in the mantle"),
    _q(248,"What is a mid-ocean ridge?",                                      "earth_science",2,"Underwater mountain range where tectonic plates diverge and new crust forms","Deep ocean trench formed by subduction","Flat plain on the ocean floor","Chain of volcanic islands"),
    _q(249,"What are Milankovitch cycles?",                                   "earth_science",2,"Variations in Earth's orbital parameters affecting long-term climate","Daily temperature changes on Earth's surface","Seasonal variation in solar energy","El Niño cycles in the Pacific Ocean"),

    # ── MATHEMATICS (250–299) ───────────────────────────────────────────────
    _q(250,"What is the value of π (pi) to 2 decimal places?",               "mathematics",0,"3.14","3.41","3.12","3.16"),
    _q(251,"What does the Pythagorean theorem state?",                        "mathematics",0,"In a right triangle, a² + b² = c²","In any triangle, a + b = c","The sum of all angles in a triangle is 180°","The hypotenuse is always the shortest side"),
    _q(252,"What is a prime number?",                                         "mathematics",0,"Number greater than 1 divisible only by 1 and itself","Any odd number","Number divisible by 2","Number with exactly three factors"),
    _q(253,"What is the square root of 144?",                                 "mathematics",0,"12","11","13","14"),
    _q(254,"What is the area formula for a circle?",                          "mathematics",0,"πr²","2πr","πd","r²"),
    _q(255,"What is a fraction?",                                             "mathematics",0,"Number representing part of a whole (numerator/denominator)","A number greater than 1","A negative number","A number with no decimal places"),
    _q(256,"What is the mean of a data set?",                                 "mathematics",0,"Sum of all values divided by the number of values","Middle value when sorted","Most frequently occurring value","Difference between highest and lowest values"),
    _q(257,"What distinguishes a permutation from a combination?",            "mathematics",1,"Permutations consider order; combinations do not","Combinations consider order; permutations do not","They are the same calculation","Permutations only apply to small sets"),
    _q(258,"What is a quadratic equation?",                                   "mathematics",1,"Polynomial equation of degree 2 (ax²+bx+c=0)","Equation with three variables","Any equation with an exponent","Equation of degree 3"),
    _q(259,"What does the binomial theorem describe?",                        "mathematics",2,"Algebraic expansion of powers of a binomial (a+b)ⁿ","Sum of an infinite series","Probability of two independent events","Formula for combinations only"),
    _q(260,"What is a derivative in calculus?",                               "mathematics",1,"Rate of change of a function at a given point (instantaneous slope)","Area under a curve","Sum of all values of a function","Limit of a function as x approaches infinity"),
    _q(261,"What is an integral in calculus?",                                "mathematics",1,"Area under a curve (antiderivative of a function)","Rate of change of a function","Slope of a tangent line","Maximum value of a function"),
    _q(262,"What is a logarithm?",                                            "mathematics",1,"Exponent to which a base must be raised to give a number","Square root of a number","Inverse of multiplication","Sum of a geometric series"),
    _q(263,"What is Euler's number (e) approximately equal to?",              "mathematics",1,"2.718","2.718×10³","3.142","1.618"),
    _q(264,"What is a matrix in mathematics?",                                "mathematics",1,"Rectangular array of numbers arranged in rows and columns","Single column of numbers","Equation with multiple variables","Set of all possible solutions"),
    _q(265,"What does the fundamental theorem of calculus state?",            "mathematics",2,"Differentiation and integration are inverse operations","All continuous functions have derivatives","Every integral has a unique solution","Integration always produces a constant"),
    _q(266,"What is a Fourier transform?",                                    "mathematics",2,"Mathematical tool decomposing functions into frequency components","Method for solving differential equations","Transformation of matrices","Technique for finding limits"),
    _q(267,"What is a complex number?",                                       "mathematics",2,"Number with real and imaginary parts (a + bi where i=√-1)","Any irrational number","Number with infinite decimal places","Negative number raised to a power"),
    _q(268,"What is the Riemann hypothesis about?",                           "mathematics",2,"Conjecture about distribution of zeros of the Riemann zeta function","Proof that π is irrational","Theorem about prime number gaps","Conjecture about perfect numbers"),
    _q(269,"What does Bayes' theorem describe?",                              "mathematics",2,"How to update probability given new evidence (conditional probability)","Total probability of all outcomes","Probability of independent events","Expected value of a random variable"),
    _q(270,"What distinguishes a vector from a scalar?",                      "mathematics",1,"Vectors have magnitude and direction; scalars have magnitude only","Scalars have direction; vectors do not","Vectors are always larger than scalars","Scalars are always positive; vectors can be negative"),
    _q(271,"What is a geometric series?",                                     "mathematics",1,"Series where each term is a constant multiple of the previous one","Series where terms increase by a constant amount","Series with alternating positive and negative terms","Series with decreasing terms only"),
    _q(272,"What distinguishes rational from irrational numbers?",            "mathematics",1,"Rational numbers can be expressed as p/q; irrational numbers cannot","Rational numbers are always integers","Irrational numbers are always negative","Rational numbers cannot be expressed as decimals"),
    _q(273,"What is set theory?",                                             "mathematics",1,"Branch of mathematics studying collections of objects (sets)","Study of geometric shapes","Analysis of number sequences","Study of probability and chance"),
    _q(274,"What does graph theory study?",                                   "mathematics",2,"Mathematical structures of vertices and edges (networks)","Plotting functions on coordinate axes","Properties of geometric shapes","Analysis of statistical data"),
    _q(275,"What is a normal distribution?",                                  "mathematics",1,"Symmetric bell-shaped probability distribution with mean at centre","Distribution where all outcomes are equally likely","Distribution skewed to one side","Distribution with two peaks"),
    _q(276,"What is standard deviation?",                                     "mathematics",1,"Measure of how spread out data is from the mean","The middle value in a data set","The most common value in a data set","Difference between maximum and minimum values"),
    _q(277,"What is modular arithmetic?",                                     "mathematics",2,"Arithmetic system where numbers wrap around after reaching a modulus","Arithmetic involving only prime numbers","System using only integers","Arithmetic applied to matrices"),
    _q(278,"What is the golden ratio approximately equal to?",                "mathematics",1,"1.618","1.414","2.718","3.142"),
    _q(279,"What does topology study?",                                       "mathematics",2,"Properties of spaces preserved under continuous deformations","Positions and angles of geometric shapes","Distribution of prime numbers","Relationships in graph networks"),
    _q(280,"What distinguishes a theorem from a conjecture?",                 "mathematics",1,"A theorem is proven; a conjecture is unproven but believed true","A conjecture is proven; a theorem is assumed true","Theorems apply to geometry only","Conjectures are always false"),
    _q(281,"What does Fermat's Last Theorem state?",                          "mathematics",2,"No whole number solution exists for aⁿ+bⁿ=cⁿ when n>2","Every even number is the sum of two primes","All prime numbers are odd","Every polynomial has at least one root"),
    _q(282,"What is Euler's identity?",                                       "mathematics",2,"e^(iπ) + 1 = 0","e^(iπ) = 1","e^π = i","i^π + e = 0"),
    _q(283,"What is a differential equation?",                                "mathematics",2,"Equation relating a function with its derivatives","Equation with two different variables","Equation involving matrices","Equation with no real solutions"),
    _q(284,"What is the travelling salesman problem?",                        "mathematics",2,"Finding shortest route visiting all cities exactly once (NP-hard problem)","Optimising transport schedules","Problem of maximum flow in networks","Finding the shortest path between two points"),
    _q(285,"What is the probability of rolling a 6 on a fair die?",          "mathematics",0,"1/6","1/3","1/2","1/4"),
    _q(286,"What is the sum of interior angles in a triangle?",              "mathematics",0,"180°","360°","90°","270°"),
    _q(287,"What is an arithmetic sequence?",                                 "mathematics",1,"Sequence where consecutive terms differ by a constant amount","Sequence where terms are multiplied by a constant","Sequence of prime numbers","Sequence with no pattern"),
    _q(288,"What is the chain rule in calculus?",                             "mathematics",2,"Rule for differentiating composite functions: d/dx[f(g(x))] = f'(g(x))·g'(x)","Rule for integrating products","Rule for finding limits","Method for partial differentiation"),
    _q(289,"What is a bijective function?",                                   "mathematics",2,"Function that is both injective (one-to-one) and surjective (onto)","Function with two outputs for each input","Function that maps all inputs to zero","Function defined only on integers"),
    _q(290,"What does the sine rule in trigonometry state?",                  "mathematics",2,"a/sin(A) = b/sin(B) = c/sin(C) for any triangle","sin²(A) + cos²(A) = 1","The sine of an angle equals the opposite/hypotenuse","All angles in a triangle sum to π"),
    _q(291,"What is a polynomial?",                                           "mathematics",1,"Expression with variables raised to non-negative integer powers","Equation with trigonometric functions","Expression involving logarithms","Equation with an infinite number of terms"),
    _q(292,"What is an asymptote?",                                           "mathematics",1,"Line that a curve approaches but never reaches","Point where a curve crosses the x-axis","Maximum value of a function","Slope of a curve at a given point"),
    _q(293,"What is Pascal's triangle primarily used for?",                   "mathematics",1,"Binomial coefficients and combinatorics","Prime number generation","Solving differential equations","Matrix multiplication"),
    _q(294,"What is the Fibonacci sequence?",                                 "mathematics",0,"Sequence where each number is the sum of the two preceding ones (1,1,2,3,5,8...)","Sequence of all prime numbers","Sequence of perfect squares","Sequence of powers of 2"),
    _q(295,"What does linear algebra primarily study?",                       "mathematics",2,"Vector spaces and linear transformations (matrices, systems of equations)","Study of curves and surfaces","Analysis of sequences and series","Study of prime numbers and divisibility"),
    _q(296,"What is a fractal?",                                              "mathematics",2,"Self-similar pattern that repeats at every scale","Any irregular geometric shape","Shape with exactly 4 sides","Pattern that appears random at all scales"),
    _q(297,"What is a limit in calculus?",                                    "mathematics",1,"Value that a function approaches as input approaches some value","Maximum value of a function","Point where a function is undefined","Constant added to a derivative"),
    _q(298,"What does number theory study?",                                  "mathematics",2,"Properties and relationships of integers, especially primes","Real numbers and their properties","Complex numbers and functions","Algebraic structures and symmetry"),
    _q(299,"What does the law of large numbers state?",                       "mathematics",2,"As sample size increases, sample mean approaches population mean","Larger numbers are always prime","Sum of large numbers follows normal distribution","Probability decreases as events repeat"),

    # ── COMPUTER SCIENCE (300–349) ──────────────────────────────────────────
    _q(300,"What does CPU stand for?",                                        "computer_science",0,"Central Processing Unit","Computer Processing Unit","Central Program Utility","Core Processing Unit"),
    _q(301,"What is binary code?",                                            "computer_science",0,"Number system using only 0s and 1s","Code written in two programming languages","Encrypted text","Code with two layers of security"),
    _q(302,"What is an algorithm?",                                           "computer_science",0,"Step-by-step procedure to solve a problem","Programming language syntax","Type of computer memory","Hardware component"),
    _q(303,"What does RAM stand for?",                                        "computer_science",0,"Random Access Memory","Read Access Memory","Rapid Application Module","Random Application Memory"),
    _q(304,"What is the internet?",                                           "computer_science",0,"Global network of interconnected computers","Single computer system","Private company network","Wireless communication standard"),
    _q(305,"What is a programming language?",                                 "computer_science",0,"Formal language used to write instructions for computers","Human natural language","Language for database queries only","System for encrypting data"),
    _q(306,"What is a variable in programming?",                              "computer_science",0,"Named storage location holding a value that can change","Fixed constant in a program","Type of programming loop","Function that returns a value"),
    _q(307,"What is a loop in programming?",                                  "computer_science",0,"Control structure that repeats code until a condition is met","Function that calls itself","Decision-making structure","Way of storing multiple values"),
    _q(308,"What is object-oriented programming?",                            "computer_science",1,"Programming paradigm organising code into objects with data and methods","Programming using only functions","Programming without loops","Style using only mathematical operations"),
    _q(309,"What is recursion in programming?",                               "computer_science",1,"Function calling itself with a simpler version of the problem","Loop that runs infinitely","Function with multiple parameters","Program running on multiple cores"),
    _q(310,"What does Big O notation describe?",                              "computer_science",1,"Upper bound on an algorithm's time or space complexity as input grows","Exact running time of an algorithm","Memory usage of a specific program","Speed of a specific processor"),
    _q(311,"What distinguishes a stack from a queue?",                        "computer_science",1,"Stack is LIFO (last in, first out); queue is FIFO (first in, first out)","Stack stores numbers; queue stores text","Stack is faster than queue","Queue is LIFO; stack is FIFO"),
    _q(312,"What is a hash table?",                                           "computer_science",1,"Data structure mapping keys to values using a hash function for fast lookup","Type of sorting algorithm","Method for compressing data","Encrypted database structure"),
    _q(313,"What is machine learning?",                                       "computer_science",1,"Systems learning from data to improve performance without explicit programming","Teaching robots to walk","Programming computers using natural language","Automated software testing"),
    _q(314,"What is a neural network?",                                       "computer_science",1,"Computing system loosely modelled on the brain with interconnected nodes","Network of computers in a building","Type of database structure","Diagram showing program flow"),
    _q(315,"What is a binary search tree?",                                   "computer_science",1,"Tree where left children are smaller and right children are larger than parent","Tree where all nodes have exactly two children","Tree sorted by insertion order","Tree used only for text data"),
    _q(316,"What is the P vs NP problem?",                                    "computer_science",2,"Whether problems verifiable in polynomial time are also solvable in polynomial time","Whether computers can simulate human intelligence","Whether encryption can always be broken","Whether all programs can be parallelised"),
    _q(317,"What is a Turing machine?",                                       "computer_science",2,"Abstract model of computation that defines the limits of what computers can compute","Physical computer designed by Alan Turing","Early model of the internet","Machine for breaking encryption"),
    _q(318,"What is cryptography?",                                           "computer_science",1,"Science of secure communication through encoding information","Study of computer viruses","Art of writing efficient code","Method of compressing files"),
    _q(319,"What is a distributed system?",                                   "computer_science",2,"Multiple computers working together appearing as a single system","Computer with multiple processors","System that runs on the internet only","Network of computers in one location"),
    _q(320,"What distinguishes TCP from UDP?",                                "computer_science",2,"TCP guarantees delivery and order; UDP is faster but does not guarantee delivery","TCP is faster; UDP is slower","UDP is used for web; TCP for video","TCP is wireless; UDP is wired"),
    _q(321,"What is a deadlock in operating systems?",                        "computer_science",2,"State where processes block each other waiting for resources, creating infinite wait","Program that runs too slowly","Memory overflow error","Condition where CPU overheats"),
    _q(322,"What is dynamic programming?",                                    "computer_science",2,"Optimisation technique breaking problems into overlapping subproblems, storing results","Programming using runtime memory allocation","Writing code that adapts to input","Programming style using dynamic typing"),
    _q(323,"What distinguishes a compiler from an interpreter?",              "computer_science",1,"Compiler translates entire program before running; interpreter runs line by line","Compilers are slower than interpreters","Interpreters produce executable files; compilers do not","Compilers are for Python; interpreters for Java"),
    _q(324,"What is version control (e.g. Git)?",                             "computer_science",1,"System tracking changes to code, enabling collaboration and rollback","Tool for testing code automatically","Software for writing documentation","Method for deploying applications"),
    _q(325,"What is an API?",                                                 "computer_science",1,"Interface allowing different software systems to communicate","Programming language for web development","Method for storing data","Type of database"),
    _q(326,"What is a relational database?",                                  "computer_science",1,"Database organising data into tables with relationships between them","Database storing unstructured data","Database existing only in memory","Database storing files and images"),
    _q(327,"What is SQL?",                                                    "computer_science",1,"Structured Query Language for managing relational databases","Programming language for web development","Language for machine learning","System for network communication"),
    _q(328,"What is cloud computing?",                                        "computer_science",1,"Delivery of computing services over the internet rather than local hardware","Computing using wireless networks","Software running without an operating system","Storing data only on physical drives"),
    _q(329,"What is a race condition?",                                       "computer_science",2,"Bug where program outcome depends on unpredictable timing of concurrent operations","Competition between two algorithms for efficiency","Error in recursive function","Condition where a loop runs too fast"),
    _q(330,"What is the Von Neumann architecture?",                           "computer_science",2,"Computer design with single memory storing both programs and data, with a CPU and bus","Network architecture for the internet","Design of parallel processing systems","Architecture for quantum computers"),
    _q(331,"What is a graph data structure?",                                 "computer_science",1,"Collection of nodes (vertices) connected by edges, representing relationships","Visual chart of data values","Type of sorted array","Two-dimensional matrix"),
    _q(332,"What distinguishes supervised from unsupervised learning?",       "computer_science",2,"Supervised uses labelled data; unsupervised finds patterns in unlabelled data","Supervised is faster; unsupervised is more accurate","Unsupervised needs human guidance; supervised does not","Supervised is for images; unsupervised for text"),
    _q(333,"What is a virtual machine?",                                      "computer_science",1,"Software emulation of a physical computer running its own operating system","Computer without a monitor","Prototype of a physical computer","Computer running on renewable energy"),
    _q(334,"What is encryption?",                                             "computer_science",1,"Process of converting data into unreadable form to prevent unauthorised access","Compressing data to reduce file size","Copying data to a backup location","Converting analogue signals to digital"),
    _q(335,"What is a buffer overflow?",                                      "computer_science",2,"Vulnerability where writing beyond buffer boundary corrupts adjacent memory","Error from full hard drive","Slow network caused by too much traffic","Error from running too many programs"),
    _q(336,"What distinguishes HTTP from HTTPS?",                             "computer_science",1,"HTTPS encrypts data in transit; HTTP does not","HTTP is faster than HTTPS","HTTPS is for internal networks only","HTTP is newer than HTTPS"),
    _q(337,"What is multithreading?",                                         "computer_science",2,"Running multiple threads concurrently within a single process","Using multiple CPUs in one computer","Running the same program multiple times","Writing code that works on all operating systems"),
    _q(338,"What is a linked list?",                                          "computer_science",1,"Data structure where each element points to the next, allowing dynamic size","Array that automatically sorts itself","List stored in a hash table","Fixed-size collection of elements"),
    _q(339,"What is the halting problem?",                                    "computer_science",2,"Undecidable problem: no algorithm can determine if any program will halt for all inputs","Problem of finding the shortest program","Challenge of sorting in minimal time","Issue of memory allocation in recursion"),
    _q(340,"What is a sorting algorithm?",                                    "computer_science",1,"Algorithm that arranges elements in a specified order (e.g. smallest to largest)","Algorithm for searching a database","Method for compressing files","Technique for encrypting data"),
    _q(341,"What distinguishes a process from a thread?",                     "computer_science",2,"Processes have separate memory; threads share memory within a process","Processes are faster than threads","Threads have separate memory; processes share memory","Processes run on GPU; threads on CPU"),
    _q(342,"What is cache memory?",                                           "computer_science",1,"Small, fast memory storing frequently accessed data close to the CPU","Main storage for programs and files","Temporary internet storage","Backup memory for crashed programs"),
    _q(343,"What is a Boolean expression?",                                   "computer_science",0,"Expression that evaluates to true or false","Mathematical calculation","String of text characters","Number with a decimal point"),
    _q(344,"What is DNS?",                                                    "computer_science",1,"Domain Name System translating domain names to IP addresses","Direct Network Security protocol","Data Network System","Digital Naming Standard"),
    _q(345,"What is a firewall?",                                             "computer_science",1,"Security system monitoring and controlling network traffic based on rules","Antivirus software","System for backing up data","Hardware component that speeds up internet"),
    _q(346,"What does Moore's law state?",                                    "computer_science",1,"Number of transistors on a chip doubles approximately every two years","Internet speed doubles every year","Storage capacity doubles every six months","Computer prices halve every five years"),
    _q(347,"What is quantum computing?",                                      "computer_science",2,"Computing using quantum mechanical phenomena (superposition, entanglement) for processing","Computing at extremely low temperatures","Computing using light instead of electricity","Computing with biological neural networks"),
    _q(348,"What is gradient descent?",                                       "computer_science",2,"Optimisation algorithm minimising a function by iteratively moving toward steepest descent","Method for sorting large datasets","Technique for compressing neural networks","Algorithm for finding the maximum of a function"),
    _q(349,"What are convolutional neural networks primarily used for?",      "computer_science",2,"Image recognition and processing tasks","Natural language processing only","Time series prediction","Reinforcement learning problems"),

    # ── HISTORY OF SCIENCE (350–399) ────────────────────────────────────────
    _q(350,"Who developed the theory of general relativity?",                 "history_of_science",0,"Albert Einstein","Isaac Newton","Niels Bohr","Max Planck"),
    _q(351,"Who discovered penicillin?",                                      "history_of_science",0,"Alexander Fleming","Louis Pasteur","Robert Koch","Joseph Lister"),
    _q(352,"Who proposed the heliocentric model of the solar system?",        "history_of_science",0,"Nicolaus Copernicus","Galileo Galilei","Johannes Kepler","Tycho Brahe"),
    _q(353,"Who is associated with the legend of discovering gravity via an apple?","history_of_science",0,"Isaac Newton","Gottfried Leibniz","Robert Hooke","Galileo Galilei"),
    _q(354,"Who co-discovered the double helix structure of DNA?",            "history_of_science",0,"Watson and Crick","Darwin and Wallace","Curie and Rutherford","Mendel and Morgan"),
    _q(355,"What was Galileo Galilei's major contribution to astronomy?",     "history_of_science",0,"Used telescope to observe moons of Jupiter and support heliocentrism","Discovered the law of gravitation","Developed the theory of relativity","Proposed the heliocentric model"),
    _q(356,"What did Marie Curie discover?",                                  "history_of_science",0,"Polonium and radium (and pioneered research on radioactivity)","Structure of the atom","Penicillin","X-rays"),
    _q(357,"Who developed the periodic table of elements?",                   "history_of_science",0,"Dmitri Mendeleev","Antoine Lavoisier","John Dalton","Humphry Davy"),
    _q(358,"What was Charles Darwin's major contribution to science?",        "history_of_science",0,"Theory of evolution by natural selection","Discovery of genetics and inheritance","Classification of all known species","Discovery of the cell"),
    _q(359,"What did Louis Pasteur contribute to medicine?",                  "history_of_science",0,"Germ theory of disease and development of vaccines","Discovery of penicillin","First successful human surgery","Development of anaesthesia"),
    _q(360,"What was the scientific revolution (roughly 16th–17th century)?", "history_of_science",1,"Shift from ancient authority to empirical observation and mathematical description of nature","Political revolution in Europe","Industrial transformation of manufacturing","Philosophical movement rejecting religion"),
    _q(361,"What was the significance of the Michelson-Morley experiment?",   "history_of_science",2,"Showed no evidence of the luminiferous ether, supporting special relativity","First measurement of the speed of light","Proved that light travels as waves","Demonstrated quantum effects of light"),
    _q(362,"What did Gregor Mendel discover?",                                "history_of_science",1,"Laws of genetic inheritance through pea plant experiments","Structure of DNA","Theory of evolution","Process of cell division"),
    _q(363,"What was the Manhattan Project?",                                 "history_of_science",1,"WWII programme that developed the first nuclear weapons","Space programme that landed men on the Moon","Cold War satellite programme","Project to build the first electronic computer"),
    _q(364,"What did Alan Turing contribute to computing?",                   "history_of_science",1,"Foundational theory of computation and codebreaking during WWII","Invented the first electronic computer","Developed the first programming language","Created the World Wide Web"),
    _q(365,"What was the significance of Sputnik in 1957?",                   "history_of_science",1,"First artificial satellite launched into orbit, beginning the space age","First spacecraft to reach the Moon","First human in space","First satellite photograph of Earth"),
    _q(366,"What did Nikola Tesla contribute to science?",                    "history_of_science",1,"Development of alternating current (AC) electrical systems","Invention of the telephone","Discovery of X-rays","Development of the internal combustion engine"),
    _q(367,"What is the significance of the Human Genome Project?",           "history_of_science",2,"First complete mapping of the human DNA sequence, completed in 2003","First successful cloning of a mammal","Discovery of DNA's double helix structure","Development of CRISPR gene editing"),
    _q(368,"What did Edwin Hubble discover?",                                 "history_of_science",1,"Galaxies beyond the Milky Way and that the universe is expanding","Existence of black holes","Theory of the Big Bang","Composition of stars via spectroscopy"),
    _q(369,"What was the Apollo 11 mission?",                                 "history_of_science",0,"First crewed mission to land on the Moon (July 1969)","First human in space","First space station","First mission to Mars"),
    _q(370,"What was Rosalind Franklin's contribution to science?",           "history_of_science",2,"X-ray crystallography images crucial to discovering DNA's double helix structure","Co-discovery of the electron","Development of the first vaccine","Discovery of radioactivity"),
    _q(371,"What did Max Planck contribute to physics?",                      "history_of_science",2,"Originated quantum theory by proposing energy is emitted in discrete packets (quanta)","Developed the theory of relativity","Discovered the electron","Proposed the nuclear model of the atom"),
    _q(372,"What was the significance of discovering the Higgs boson in 2012?","history_of_science",2,"Confirmed the Standard Model's mechanism for giving particles mass","Proved the existence of dark matter","Confirmed the Big Bang theory","Proved quantum entanglement at large scales"),
    _q(373,"What were Archimedes' main contributions to science?",            "history_of_science",1,"Principles of buoyancy, levers, and early calculus concepts","Heliocentric model of solar system","Laws of planetary motion","Theory of light and optics"),
    _q(374,"What did Copernicus challenge with his heliocentric model?",      "history_of_science",1,"The prevailing geocentric (Earth-centred) view of the universe","Newton's laws of gravity","The atomic theory of matter","The existence of other galaxies"),
    _q(375,"What was the significance of the first successful smallpox vaccination?","history_of_science",1,"Demonstrated vaccines could prevent disease, founding immunology (Edward Jenner, 1796)","First use of antibiotics","First organ transplant","First use of anaesthesia in surgery"),
    _q(376,"What did James Clerk Maxwell unify?",                             "history_of_science",2,"Electricity, magnetism, and light into a single theory of electromagnetism","Gravity and electromagnetism","Quantum mechanics and relativity","Thermodynamics and mechanics"),
    _q(377,"Who is credited with inventing the telephone?",                   "history_of_science",0,"Alexander Graham Bell","Thomas Edison","Nikola Tesla","Guglielmo Marconi"),
    _q(378,"What did Ibn al-Haytham (Alhazen) contribute to science?",       "history_of_science",2,"Pioneered scientific method and optics (how eyes and light work, ~1000 CE)","Discovery of algebra","Development of trigonometry","First accurate star catalogue"),
    _q(379,"What did Robert Hooke discover using a microscope?",              "history_of_science",1,"Cells (he named them after observing cork tissue)","Bacteria","DNA structure","Red blood cells"),
    _q(380,"What was the significance of Boyle's Law?",                       "history_of_science",1,"Established that pressure and volume of a gas are inversely proportional at constant temperature","First law of thermodynamics","Relationship between temperature and gas pressure","Discovery of oxygen"),
    _q(381,"What did Antoine Lavoisier establish?",                           "history_of_science",2,"Law of conservation of mass and modern chemical nomenclature","Atomic theory of matter","Periodic table of elements","Theory of combustion using phlogiston"),
    _q(382,"Who independently invented calculus (disputed priority)?",        "history_of_science",1,"Newton and Leibniz","Euler and Gauss","Fermat and Pascal","Archimedes and Euclid"),
    _q(383,"What was the significance of Michael Faraday's work?",            "history_of_science",2,"Discovered electromagnetic induction, forming the basis of electric motors and generators","Unified electricity and light mathematically","Developed the first battery","Discovered the electron"),
    _q(384,"What did Rachel Carson's 'Silent Spring' (1962) achieve?",        "history_of_science",1,"Raised awareness of pesticide dangers and helped launch the environmental movement","Discovered the greenhouse effect","Proved climate change was human-caused","Led to creation of NASA"),
    _q(385,"What is the significance of the Voyager probes?",                 "history_of_science",1,"First spacecraft to travel to outer solar system planets and enter interstellar space","First probes to land on Mars","First satellites to photograph Earth from space","First spacecraft to reach the Moon"),
    _q(386,"What did Barbara McClintock discover?",                           "history_of_science",2,"Transposable elements (jumping genes) in maize — genetic sequences that move","Structure of DNA double helix","Mechanism of cell division","RNA's role in protein synthesis"),
    _q(387,"What was the significance of discovering X-rays (Röntgen, 1895)?","history_of_science",1,"Enabled non-invasive imaging of the body's interior, revolutionising medicine","Proved light is a wave","Led to discovery of radioactivity","First evidence of atomic structure"),
    _q(388,"What did Mendeleev predict using his periodic table?",            "history_of_science",2,"Properties of undiscovered elements (which were later found and confirmed)","Existence of subatomic particles","Structure of the atom","Number of elements that would ever exist"),
    _q(389,"What is the legacy of the Large Hadron Collider (LHC)?",          "history_of_science",2,"Discovery of Higgs boson (2012) and testing of fundamental physics at high energies","First nuclear fission reaction","Development of nuclear power","Discovery of the neutron"),
    _q(390,"What did John Snow do during the 1854 cholera outbreak?",         "history_of_science",2,"Used data mapping to trace cholera to a water pump, founding epidemiology","Developed the first cholera vaccine","Discovered the cholera bacterium","Created the first public health laws"),
    _q(391,"What was the significance of Pasteur's germ theory?",             "history_of_science",1,"Established that microorganisms cause disease, overturning spontaneous generation","First description of viruses","Discovery of antibiotics","Development of the first surgical techniques"),
    _q(392,"Who was the first person to walk on the Moon?",                   "history_of_science",0,"Neil Armstrong","Buzz Aldrin","Yuri Gagarin","John Glenn"),
    _q(393,"What did Christiaan Huygens contribute to wave theory?",          "history_of_science",2,"Proposed light travels as waves and developed wave theory of light","Discovered the particle nature of light","Invented the first telescope","Developed the theory of diffraction"),
    _q(394,"What was significant about the Wright Brothers' achievement?",    "history_of_science",0,"First successful powered, controlled aeroplane flight (1903)","First jet-powered aircraft","First transatlantic flight","First helicopter flight"),
    _q(395,"What did Jonas Salk develop?",                                    "history_of_science",1,"First effective polio vaccine (1955)","First antibiotic (penicillin)","First cancer treatment","Smallpox vaccine"),
    _q(396,"What is the significance of CERN?",                              "history_of_science",2,"World's largest particle physics lab; discovered Higgs boson and invented the World Wide Web","First nuclear power plant","Organisation that launched first satellites","Lab that developed the first computer"),
    _q(397,"What did Carl Sagan contribute to science?",                      "history_of_science",1,"Pioneered science communication and contributed to planetary science and SETI","Discovered the structure of DNA","Developed the theory of cosmic inflation","Founded the field of astrobiology"),
    _q(398,"What was significant about Einstein's 1905 'miracle year'?",      "history_of_science",2,"Published special relativity, photoelectric effect, Brownian motion, and E=mc² in one year","Developed general relativity and unified gravity","First proved quantum mechanics experimentally","Received the Nobel Prize for relativity"),
    _q(399,"What did James Watt improve that helped start the Industrial Revolution?","history_of_science",1,"Steam engine (adding separate condenser, greatly improving efficiency)","Cotton-spinning machine","Iron-smelting process","Railway locomotive"),

    # ── ENVIRONMENTAL SCIENCE (400–449) ─────────────────────────────────────
    _q(400,"What is climate change?",                                         "environmental_science",0,"Long-term shifts in global temperatures and weather patterns, largely driven by human activities","Day-to-day weather variations","Seasonal changes in temperature","Natural cycles of ice ages only"),
    _q(401,"What is the ozone layer?",                                        "environmental_science",0,"Region of stratosphere containing ozone (O₃) that absorbs harmful UV radiation","Layer of clouds in the troposphere","Layer of CO₂ trapping heat","Outer boundary of Earth's atmosphere"),
    _q(402,"What is deforestation?",                                          "environmental_science",0,"Large-scale removal of forests, often for agriculture or development","Natural death of trees in winter","Process of reforestation","Controlled burning of forest undergrowth"),
    _q(403,"What is a renewable energy source?",                              "environmental_science",0,"Energy from naturally replenishing sources (solar, wind, water)","Energy from coal and oil","Energy from nuclear fission only","Energy stored in batteries"),
    _q(404,"What causes acid rain?",                                          "environmental_science",1,"Sulphur dioxide and nitrogen oxides from burning fossil fuels reacting with atmospheric water","Increased CO₂ in the atmosphere","Evaporation of acidic ocean water","Industrial chemicals released directly into rain clouds"),
    _q(405,"What is a carbon footprint?",                                     "environmental_science",0,"Total greenhouse gas emissions caused by an individual, organisation, or product","Amount of carbon stored in a forest","Size of a company's factory","Amount of CO₂ in the atmosphere"),
    _q(406,"What is biodegradable waste?",                                    "environmental_science",0,"Waste that can be broken down by microorganisms into natural substances","Waste that can be recycled into new products","Hazardous chemical waste","Waste that can never decompose"),
    _q(407,"What is the Paris Agreement?",                                    "environmental_science",1,"International treaty (2015) to limit global warming to well below 2°C above pre-industrial levels","Treaty banning nuclear weapons","Agreement on ocean fishing rights","Protocol for reducing ozone-depleting substances"),
    _q(408,"What is desertification?",                                        "environmental_science",1,"Process by which fertile land becomes desert due to drought, deforestation, or poor land use","Natural expansion of existing deserts","Formation of sand dunes by wind","Seasonal drying of river beds"),
    _q(409,"What is an invasive species?",                                    "environmental_science",1,"Non-native organism causing harm to the environment or native species in a new habitat","Endangered species protected by law","Species unique to one geographic area","Predator species at the top of a food chain"),
    _q(410,"What is eutrophication?",                                         "environmental_science",2,"Excessive nutrient enrichment of water causing algal blooms and oxygen depletion","Natural ageing of a lake ecosystem","Warming of ocean water due to climate change","Increase in water salinity over time"),
    _q(411,"What distinguishes climate change mitigation from adaptation?",   "environmental_science",2,"Mitigation reduces causes; adaptation adjusts to impacts","Mitigation is short-term; adaptation is long-term","Adaptation addresses CO₂; mitigation addresses temperature","They mean the same thing"),
    _q(412,"What is ocean acidification?",                                    "environmental_science",2,"Decrease in ocean pH as oceans absorb CO₂ from the atmosphere","Increase in ocean salinity due to evaporation","Warming of ocean water causing coral bleaching","Pollution of oceans with industrial acids"),
    _q(413,"What is biomagnification?",                                       "environmental_science",2,"Increasing concentration of toxins in organisms at higher trophic levels","Equal distribution of toxins throughout a food chain","Breakdown of toxins by microorganisms","Dilution of pollutants in large water bodies"),
    _q(414,"What is the difference between point and non-point source pollution?","environmental_science",2,"Point source is identifiable (e.g. a pipe); non-point source is diffuse (e.g. agricultural runoff)","Point source is more dangerous","Non-point source is easier to regulate","Point source only occurs in cities"),
    _q(415,"What is the tragedy of the commons?",                             "environmental_science",2,"Individuals overuse shared resources for self-interest, depleting them for everyone","Problem of government ownership of land","Issue of too many conservation laws","Conflict between urban and rural land use"),
    _q(416,"What is soil degradation?",                                       "environmental_science",1,"Decline in soil quality due to erosion, nutrient depletion, or contamination","Natural weathering of rocks into soil","Formation of new soil from organic matter","Increase in soil salinity due to irrigation"),
    _q(417,"What is the IPCC?",                                               "environmental_science",1,"Intergovernmental Panel on Climate Change — UN body assessing climate science","International Pollution Control Commission","International Plant Conservation Committee","Independent Panel for Carbon Credits"),
    _q(418,"What is a carbon sink?",                                          "environmental_science",1,"Natural or artificial reservoir absorbing more CO₂ than it releases","Source of carbon emissions","Process of burning fossil fuels","Underground CO₂ storage facility"),
    _q(419,"What is coral bleaching?",                                        "environmental_science",1,"Coral expelling symbiotic algae due to stress (e.g. warming water), turning white","Natural seasonal colour change in coral","Coral dying due to ocean acidification","Process of coral reproduction"),
    _q(420,"What is the role of wetlands in ecosystems?",                     "environmental_science",1,"Water filtration, flood control, carbon storage, and biodiversity support","Producing oxygen for the atmosphere","Regulating ocean temperature","Providing farmland for agriculture"),
    _q(421,"What is bioremediation?",                                         "environmental_science",2,"Using living organisms (bacteria, plants) to remove or neutralise pollutants","Chemical treatment of contaminated soil","Physical removal of toxic waste","Burning contaminated material to destroy toxins"),
    _q(422,"What is sustainable development?",                                "environmental_science",1,"Development meeting present needs without compromising future generations' ability to meet theirs","Halting all economic growth","Development focused only on renewable energy","Economic growth without environmental consideration"),
    _q(423,"What are greenhouse gases?",                                      "environmental_science",0,"Gases (CO₂, CH₄, N₂O) that trap heat in the atmosphere","All gases in the atmosphere","Gases released only by volcanoes","Gases used in refrigeration only"),
    _q(424,"What is nuclear waste and why is it a concern?",                  "environmental_science",1,"Radioactive material from nuclear reactors that remains hazardous for thousands of years","Chemical waste from factories","Carbon emissions from nuclear power","Water contaminated by uranium mining"),
    _q(425,"What is the Great Pacific Garbage Patch?",                        "environmental_science",1,"Large accumulation of marine debris (mainly plastic) in the North Pacific Ocean","Volcanic island chain in the Pacific","Dead zone in the Pacific with no marine life","Radioactive zone from nuclear testing"),
    _q(426,"What are microplastics and why are they a concern?",              "environmental_science",1,"Tiny plastic particles (<5mm) that accumulate in ecosystems and organisms, causing harm","Biodegradable plastic substitute","Small plastic recycling pellets","Plastic coating on electronic components"),
    _q(427,"What is carbon sequestration?",                                   "environmental_science",2,"Process of capturing and storing CO₂ from the atmosphere long-term","Measuring carbon emissions from industry","Converting CO₂ to oxygen using plants","Reducing carbon in soil through farming"),
    _q(428,"What is the relationship between biodiversity and ecosystem stability?","environmental_science",2,"Higher biodiversity generally increases resilience and stability of ecosystems","Biodiversity has no effect on ecosystem function","More species always reduces ecosystem productivity","Stability requires few but dominant species"),
    _q(429,"What is agroforestry?",                                           "environmental_science",2,"Integrating trees with crops or livestock for environmental and economic benefits","Cutting forests for agricultural land","Large-scale single-crop farming","Vertical farming in urban areas"),
    _q(430,"What is the nitrogen pollution problem?",                         "environmental_science",2,"Excess nitrogen from fertilisers causing eutrophication, dead zones, and air pollution","Depletion of nitrogen from the atmosphere","Nitrogen causing ozone layer depletion","Toxic nitrogen gas from vehicle emissions"),
    _q(431,"What is the environmental Kuznets curve?",                        "environmental_science",2,"Hypothesis that pollution initially increases then decreases as income rises (inverted U)","Curve showing global temperature rise over time","Graph of species extinction rates","Relationship between CO₂ and ocean temperature"),
    _q(432,"What is a circular economy?",                                     "environmental_science",1,"Economic system aiming to eliminate waste by keeping resources in use as long as possible","Economy based on circular trade routes","System where only renewable energy is used","Financial model for environmental companies"),
    _q(433,"What is soil salinisation?",                                      "environmental_science",2,"Accumulation of salt in soil reducing fertility, often from irrigation","Natural salt content in coastal soils","Process of removing salt from seawater","Erosion of soil by saltwater flooding"),
    _q(434,"What is a keystone species?",                                     "environmental_science",1,"Species with disproportionately large effect on its ecosystem relative to its abundance","Most numerous species in an ecosystem","Species at the top of the food chain","Endangered species protected by law"),
    _q(435,"What is the precautionary principle in environmental policy?",    "environmental_science",2,"When an action risks harm, precautionary measures should be taken even without full scientific certainty","Policy requiring proof of harm before any action","Principle that economic growth must precede environmental protection","Requirement to always choose the cheapest solution"),
    _q(436,"What is the role of phytoplankton in the ocean?",                "environmental_science",1,"Base of marine food webs and responsible for ~50% of Earth's oxygen production","Top predators in marine ecosystems","Filter feeders that clean ocean water","Primary consumers of marine algae"),
    _q(437,"What is habitat fragmentation?",                                  "environmental_science",1,"Breaking up of continuous habitat into smaller isolated patches, reducing biodiversity","Complete destruction of a habitat","Seasonal change in habitat availability","Migration of animals between habitats"),
    _q(438,"What distinguishes conservation from preservation?",              "environmental_science",1,"Conservation manages sustainable use; preservation protects from all human use","Conservation bans all human access","Preservation allows sustainable harvesting","They mean the same thing"),
    _q(439,"What is an ecosystem service?",                                   "environmental_science",1,"Benefit that humans receive from ecosystems (e.g. clean water, pollination, climate regulation)","Service provided by environmental agencies","Paid work in conservation","Tourism revenue from national parks"),
    _q(440,"What is green infrastructure?",                                   "environmental_science",2,"Network of natural and semi-natural areas providing ecosystem services in urban and rural settings","Renewable energy installations","Electric vehicle charging networks","Eco-friendly building materials"),
    _q(441,"What is light pollution?",                                        "environmental_science",0,"Excessive artificial light at night disrupting ecosystems and obscuring the night sky","UV radiation from the Sun","Pollution caused by nuclear power plants","Radiation from electronic devices"),
    _q(442,"What is water scarcity?",                                         "environmental_science",0,"Lack of sufficient available water resources to meet water needs in a region","Flooding caused by heavy rainfall","Contamination of water supplies","Evaporation of ocean water"),
    _q(443,"What is an ecological footprint?",                                "environmental_science",1,"Amount of biologically productive land and water needed to sustain a person's lifestyle","Total area of a person's land ownership","Carbon emissions of one individual per year","Amount of water used by one person annually"),
    _q(444,"What causes the difference between solar and wind energy?",       "environmental_science",0,"Solar converts sunlight to electricity; wind uses turbines driven by moving air","Solar is more efficient in all conditions","Wind works during day only; solar works at night","They use the same conversion technology"),
    _q(445,"What is the difference between endangered and extinct species?",  "environmental_science",0,"Endangered are at risk of extinction but still exist; extinct no longer exist","Extinct species can be revived; endangered cannot","Endangered species are legally protected; extinct are not","They mean the same thing"),
    _q(446,"What is sustainable development goal (SDG) 13?",                  "environmental_science",1,"Climate Action — urgent action to combat climate change and its impacts","Zero Hunger","Clean Water and Sanitation","Life Below Water"),
    _q(447,"What is the difference between conservation and preservation?",   "environmental_science",1,"Conservation allows sustainable use; preservation protects from all use","Conservation is stricter than preservation","Preservation allows some human use; conservation does not","They are synonymous in modern usage"),
    _q(448,"What is green infrastructure?",                                   "environmental_science",2,"Network of natural areas providing ecosystem services alongside built infrastructure","Only solar panels and wind turbines","Eco-friendly concrete and steel","Underground waste management systems"),
    _q(449,"What is the role of wetlands in the carbon cycle?",               "environmental_science",2,"Wetlands store large amounts of carbon in peat and vegetation, acting as carbon sinks","Wetlands release more CO₂ than any other ecosystem","Wetlands have no significant role in carbon storage","Wetlands only store carbon in their water"),

    # ── HUMAN BODY (450–499) ─────────────────────────────────────────────────
    _q(450,"How many bones are in the adult human body?",                     "human_body",0,"206","208","212","198"),
    _q(451,"What is the largest organ in the human body?",                    "human_body",0,"Skin","Liver","Brain","Lungs"),
    _q(452,"What does the heart do?",                                         "human_body",0,"Pumps blood throughout the body","Filters waste from blood","Produces hormones","Controls breathing"),
    _q(453,"What is the main function of the lungs?",                         "human_body",0,"Gas exchange — absorbing oxygen and expelling carbon dioxide","Filtering blood","Producing white blood cells","Regulating body temperature"),
    _q(454,"What are the four main components of blood?",                     "human_body",0,"Red blood cells, white blood cells, platelets, plasma","Plasma, hormones, enzymes, water","Red cells, bone marrow, salt, oxygen","White cells, haemoglobin, fibrin, lipids"),
    _q(455,"What is the main role of the brain?",                             "human_body",0,"Control centre for thought, movement, sensation, and body functions","Produce hormones for the body","Filter toxins from blood","Store energy for the body"),
    _q(456,"What is the main function of the kidneys?",                       "human_body",0,"Filter blood and produce urine to remove waste products","Produce digestive enzymes","Regulate blood sugar levels","Produce red blood cells"),
    _q(457,"What is the nervous system?",                                     "human_body",0,"Network of nerves and cells transmitting signals throughout the body","System of blood vessels","System of glands producing hormones","Digestive tract and associated organs"),
    _q(458,"What does the digestive system do?",                              "human_body",0,"Breaks down food and absorbs nutrients into the bloodstream","Filters waste from the blood","Produces hormones regulating body functions","Transports oxygen to cells"),
    _q(459,"What is the immune system?",                                      "human_body",0,"Body's defence system against pathogens and disease","System of blood vessels","Network of nerves","System producing hormones"),
    _q(460,"What is the main difference between arteries and veins?",         "human_body",1,"Arteries carry oxygenated blood away from heart; veins carry deoxygenated blood back (except pulmonary)","Arteries are larger than veins","Veins carry blood away from heart; arteries return it","Arteries are only in the legs; veins are elsewhere"),
    _q(461,"What is the endocrine system?",                                   "human_body",1,"System of glands producing hormones that regulate body functions","System of nerves controlling movement","Digestive organs and glands","System filtering blood and producing urine"),
    _q(462,"What is the main function of the liver?",                         "human_body",1,"Detoxification, metabolism, bile production, and nutrient processing","Filtering blood and producing urine","Absorbing oxygen from the lungs","Producing antibodies"),
    _q(463,"What is the role of white blood cells?",                          "human_body",1,"Defend the body against infection and foreign substances","Carry oxygen around the body","Clot blood at wound sites","Transport nutrients from the intestines"),
    _q(464,"What distinguishes the central from the peripheral nervous system?","human_body",1,"CNS comprises brain and spinal cord; PNS is all other nerves","CNS is voluntary; PNS is involuntary","CNS controls digestion; PNS controls movement","CNS is in the limbs; PNS is in the torso"),
    _q(465,"What is the musculoskeletal system?",                             "human_body",1,"System of muscles, bones, joints, and connective tissues enabling movement","System of nerves controlling muscles","Digestive organs and support structures","System of blood vessels supplying muscles"),
    _q(466,"What is a synapse?",                                              "human_body",2,"Junction between two neurons where signals are transmitted chemically or electrically","Cell body of a neuron","Myelin sheath around a nerve","Receptor on a muscle cell"),
    _q(467,"What is the role of the hypothalamus?",                           "human_body",2,"Links nervous system to endocrine system; regulates body temperature, hunger, sleep","Controls conscious thought and decision-making","Processes visual and auditory information","Regulates heart rate directly"),
    _q(468,"What is the blood-brain barrier?",                                "human_body",2,"Selectively permeable barrier protecting the brain from harmful substances in blood","Membrane separating left and right brain hemispheres","Protective layer of cerebrospinal fluid","Physical barrier of the skull"),
    _q(469,"What is autoimmune disease?",                                     "human_body",2,"Condition where immune system attacks the body's own healthy cells","Allergy to environmental substances","Disease caused by vitamin deficiency","Infection that the immune system cannot fight"),
    _q(470,"What is homeostasis in the human body?",                          "human_body",1,"Maintenance of stable internal conditions (temperature, pH, blood glucose)","Growth in response to exercise","Adaptation to external environment","Immune response to infection"),
    _q(471,"What distinguishes Type 1 from Type 2 diabetes?",                "human_body",2,"Type 1: autoimmune destruction of insulin-producing cells; Type 2: insulin resistance","Type 1 is caused by diet; Type 2 is genetic","Type 2 requires insulin injections; Type 1 does not","Type 1 is more common than Type 2"),
    _q(472,"What is the role of insulin?",                                    "human_body",1,"Allows cells to absorb glucose from blood, lowering blood sugar levels","Breaks down fats for energy","Stimulates production of red blood cells","Controls body temperature"),
    _q(473,"What is the lymphatic system?",                                   "human_body",1,"Network of vessels and organs maintaining fluid balance and supporting immune function","System of arteries and veins","Network of digestive organs","System of hormone-producing glands"),
    _q(474,"What are stem cells in the human body?",                          "human_body",1,"Undifferentiated cells that can develop into various specialised cell types","Cells that produce hormones","Mature red blood cells","Cells specialised for nerve signal transmission"),
    _q(475,"What is the function of the spleen?",                             "human_body",2,"Filters blood, recycles red blood cells, and supports immune function","Produces digestive enzymes","Regulates blood glucose","Produces bile for digestion"),
    _q(476,"What distinguishes a neuron from a nerve?",                       "human_body",2,"A neuron is a single cell; a nerve is a bundle of many neurons' axons","Neurons are in the brain; nerves are in the body","Nerves send signals; neurons receive them","Neurons are larger than nerves"),
    _q(477,"What is the role of the pancreas?",                               "human_body",1,"Produces insulin/glucagon (hormones) and digestive enzymes","Filters blood and produces urine","Absorbs nutrients from digested food","Produces bile for fat digestion"),
    _q(478,"What is cellular respiration?",                                   "human_body",1,"Process converting glucose and oxygen into ATP energy (plus CO₂ and water)","Exchange of gases in the lungs","Absorption of oxygen into the blood","Breakdown of food in the stomach"),
    _q(479,"What is the main role of the small intestine?",                   "human_body",1,"Absorb nutrients from digested food into the bloodstream","Break down food mechanically","Absorb water and compact waste","Produce digestive enzymes only"),
    _q(480,"What distinguishes ligaments from tendons?",                      "human_body",1,"Ligaments connect bone to bone; tendons connect muscle to bone","Tendons connect bone to bone; ligaments connect muscle to bone","Ligaments are in joints; tendons are in muscles only","They are the same structure"),
    _q(481,"What is the role of the thyroid gland?",                          "human_body",1,"Produces hormones regulating metabolism, growth, and energy production","Produces insulin to regulate blood sugar","Controls the immune system","Regulates heart rate directly"),
    _q(482,"What is an antibody?",                                            "human_body",1,"Protein produced by B cells that specifically binds to and neutralises antigens","White blood cell that destroys pathogens","Enzyme that breaks down bacteria","Hormone that stimulates immune response"),
    _q(483,"What is the function of the cerebellum?",                         "human_body",2,"Coordinates movement, balance, and fine motor control","Controls speech and language","Regulates emotions and memory","Controls unconscious breathing and heart rate"),
    _q(484,"What is the autonomic nervous system?",                           "human_body",2,"Part of PNS controlling involuntary functions (heart rate, digestion, breathing rate)","System controlling voluntary muscle movements","Network of sensory nerves in the skin","Nerves controlling conscious thought"),
    _q(485,"What distinguishes somatic from visceral pain?",                  "human_body",2,"Somatic pain originates from skin/muscles/bones; visceral pain from internal organs","Somatic pain is more severe","Visceral pain is easier to localise","Somatic pain is always chronic; visceral is acute"),
    _q(486,"What is the role of the adrenal glands?",                         "human_body",2,"Produce adrenaline, cortisol and other hormones regulating stress and metabolism","Produce insulin and glucagon","Filter waste from the blood","Regulate calcium levels in blood"),
    _q(487,"What is the complement system in immunity?",                      "human_body",2,"Group of proteins that enhance ability of antibodies to clear pathogens from the body","System of complementary DNA strands","Backup immune response using T cells only","Set of enzymes digesting foreign proteins in blood"),
    _q(488,"What distinguishes innate from acquired immunity?",               "human_body",1,"Innate is non-specific and immediate; acquired is specific and develops after exposure","Innate involves antibodies; acquired does not","Acquired immunity is present from birth","Innate immunity requires vaccination"),
    _q(489,"What is peristalsis?",                                            "human_body",1,"Wave-like muscle contractions moving food through the digestive tract","Filtering of blood by the kidneys","Expansion of lungs during breathing","Pumping action of the heart"),
    _q(490,"What is the role of bile in digestion?",                          "human_body",1,"Emulsifies fats in the small intestine to aid their digestion and absorption","Breaks down proteins into amino acids","Neutralises stomach acid in the intestine","Absorbs water from digested food"),
    _q(491,"What is a hormone?",                                              "human_body",0,"Chemical messenger produced by glands and transported in blood to target organs","Type of enzyme in the digestive system","Protein in the immune system","Neurotransmitter in the brain"),
    _q(492,"What is the typical resting heart rate for a healthy adult?",     "human_body",0,"60–100 beats per minute","30–50 beats per minute","100–140 beats per minute","120–160 beats per minute"),
    _q(493,"What is the function of platelets?",                              "human_body",1,"Help form blood clots to stop bleeding at wound sites","Carry oxygen around the body","Fight bacterial infections","Transport hormones in the blood"),
    _q(494,"What is the role of the corpus callosum?",                        "human_body",2,"Connects left and right cerebral hemispheres, enabling communication between them","Controls balance and coordination","Regulates the autonomic nervous system","Produces cerebrospinal fluid"),
    _q(495,"What distinguishes osteoporosis from osteoarthritis?",            "human_body",2,"Osteoporosis: reduced bone density; Osteoarthritis: degeneration of joint cartilage","Osteoarthritis affects bones; osteoporosis affects joints","Both conditions are the same disease","Osteoporosis only affects the spine"),
    _q(496,"What is the role of the retina in vision?",                       "human_body",1,"Contains photoreceptors (rods and cones) that convert light into nerve signals","Focuses light onto the back of the eye","Controls amount of light entering the eye","Protects the eye from physical damage"),
    _q(497,"What distinguishes motor neurons from sensory neurons?",          "human_body",2,"Motor neurons carry signals from CNS to muscles; sensory neurons carry signals from receptors to CNS","Sensory neurons are in muscles; motor neurons are in skin","Motor neurons are larger than sensory neurons","They perform identical functions"),
    _q(498,"What is the glomerulus in the kidney?",                           "human_body",2,"Cluster of capillaries in the nephron where blood is filtered","Collecting duct for urine","Valve controlling urine flow","Loop where water is reabsorbed"),
    _q(499,"What is the role of myelin in the nervous system?",               "human_body",2,"Insulating sheath around axons that speeds up nerve signal transmission","Produces neurotransmitters at synapses","Connects neurons to muscle fibres","Protects cell body of a neuron from damage"),
]

import random

def shuffle_choices(question, seed=None):
    """Return question with choices shuffled (correct answer in random position)."""
    rng = random.Random(seed if seed is not None else question["id"])
    choices = question["choices"][:]
    rng.shuffle(choices)
    return {**question, "choices": choices}

N_ACTIONS = len(QUESTIONS)

assert len(QUESTIONS) == 500
assert len({q["id"] for q in QUESTIONS}) == 500
assert all(q["difficulty"] in (0, 1, 2) for q in QUESTIONS)
assert all(q["topic"] in TOPICS for q in QUESTIONS)
assert all(len(q["choices"]) == 4 for q in QUESTIONS)
assert all(q["correct"] in q["choices"] for q in QUESTIONS)


# ═══════════════════════════════════════════════════════════════
# STUDENT SIMULATOR & RL ENVIRONMENT
# ═══════════════════════════════════════════════════════════════
# environment.py — inlined (questions defined above in notebook)
# All QUESTIONS, TOPICS, DIFFICULTY_NAMES, N_ACTIONS already in scope

# environment.py — inlined (questions defined above in notebook)
# All QUESTIONS, TOPICS, DIFFICULTY_NAMES, N_ACTIONS already in scope

# environment.py — inlined (questions defined above in notebook)
# All QUESTIONS, TOPICS, DIFFICULTY_NAMES, N_ACTIONS already in scope

import numpy as np


# ─────────────────────────────────────────────────────────────
# STUDENT SIMULATOR
# ─────────────────────────────────────────────────────────────

## Cell 3 — RL Environment: Student Simulator

### `StudentSimulator`
Models a realistic student with:
- **Per-topic knowledge** (0.0–1.0) — starts random, grows with each answer
- **Per-topic learning rate** — some students learn faster in certain subjects
- **Fatigue** — accuracy degrades over a long session
- **Confidence** — boosts accuracy on streaks, drops on wrong answers

### `TutoringEnvironment`
Standard RL loop wrapping the student:
- **State**: 7-dimensional tuple encoding student context
- **Action**: integer 0–499 (which question to ask)
- **Reward**: shaped to reward difficulty matching and avoid repetition

**Reward function:**
```
Correct + difficulty matches knowledge level → +1.0 to +2.0
Correct but mismatched difficulty            → +0.4 to +1.4
Wrong answer on hard question (struggling)   → -0.9 to -1.5
Repeated question                            → -0.8 penalty
```

In [ ]:
class StudentSimulator:
    """
    Simulates a student with:
      • Per-topic knowledge  (0.0–1.0)
      • Per-topic learning rate  (how fast they absorb material)
      • Fatigue               (reduces accuracy over a long session)
      • Confidence            (boosts accuracy on easy wins, dips on failures)

    Answering correctly on a harder question gives more learning gain.
    Getting it wrong still gives a small gain (exposure effect).
    """

    def __init__(self, seed=None):
        rng = np.random.default_rng(seed)

        # Knowledge: random starting level 0.10 – 0.55 per topic
        self.knowledge = {t: float(rng.uniform(0.10, 0.55)) for t in TOPICS}

        # Learning rate: how quickly each topic's knowledge grows (0.6 – 1.4×)
        self.learning_rate = {t: float(rng.uniform(0.6, 1.4)) for t in TOPICS}

        # Fatigue: starts at 0, grows with each question
        self.fatigue = 0.0
        self.fatigue_rate = float(rng.uniform(0.005, 0.015))   # per question

        # Confidence: starts neutral
        self.confidence = float(rng.uniform(0.45, 0.65))
        self.rng = rng

        # Tracking
        self.total_answered = 0
        self.total_correct  = 0
        self.history        = []   # list of dicts

    # ──────────────────────────────────────────

    def answer(self, question: dict) -> bool:
        topic      = question["topic"]
        difficulty = question["difficulty"]

        # Base probability
        diff_penalty = [0.0, 0.18, 0.38][difficulty]
        p = self.knowledge[topic] - diff_penalty

        # Modulate by fatigue and confidence
        p = p * (1.0 - self.fatigue * 0.5)
        p = p + (self.confidence - 0.5) * 0.1
        p = float(np.clip(p, 0.04, 0.96))

        correct = bool(self.rng.random() < p)

        # Knowledge gain (harder = more gain when correct)
        base_gain = [0.020, 0.035, 0.055][difficulty]
        lr        = self.learning_rate[topic]
        if correct:
            self.knowledge[topic] = min(1.0, self.knowledge[topic] + base_gain * lr)
            self.confidence       = min(1.0, self.confidence + 0.02)
        else:
            self.knowledge[topic] = min(1.0, self.knowledge[topic] + base_gain * lr * 0.25)
            self.confidence       = max(0.1, self.confidence - 0.03)

        # Fatigue accumulates
        self.fatigue = min(0.6, self.fatigue + self.fatigue_rate)

        self.total_answered += 1
        if correct:
            self.total_correct += 1

        self.history.append({
            "qid":       question["id"],
            "topic":     topic,
            "diff":      difficulty,
            "correct":   correct,
            "knowledge": dict(self.knowledge),
            "fatigue":   self.fatigue,
            "confidence":self.confidence,
        })
        return correct

    def accuracy(self) -> float:
        return self.total_correct / max(1, self.total_answered)

    def overall_knowledge(self) -> float:
        return float(np.mean(list(self.knowledge.values())))

    def topic_knowledge(self, topic: str) -> float:
        return self.knowledge[topic]


# ─────────────────────────────────────────────────────────────
# TUTORING ENVIRONMENT
# ─────────────────────────────────────────────────────────────

class TutoringEnvironment:
    """
    RL Environment wrapping the student simulator.

    STATE (7 dimensions → encoded as tuple for Q-table, array for DQN):
        0  topic_idx        0–9   (current question's topic)
        1  difficulty       0–2
        2  acc_bucket       0=struggling(<35%), 1=ok(35-65%), 2=good(>65%)
        3  knowledge_bucket 0=low(<0.35), 1=mid(0.35-0.65), 2=high(>0.65) [current topic avg]
        4  fatigue_bucket   0=fresh(<0.2), 1=moderate(0.2-0.4), 2=tired(>0.4)
        5  session_progress 0=early(<33%), 1=mid(33-66%), 2=late(>66%)
        6  repeat_flag      0=new question, 1=repeated

    ACTION: 0–499 (which question to ask)

    REWARD: shaped to reward appropriate difficulty and novel questions
    """

    def __init__(self, student: StudentSimulator = None, max_steps: int = 40):
        self.max_steps = max_steps
        self.student   = student or StudentSimulator()
        self.reset()

    # ──────────────────────────────────────────

    def reset(self, student: StudentSimulator = None):
        if student:
            self.student = student
        self.step_count     = 0
        self.asked_ids      = set()
        self.recent_results = []
        return self._get_state(QUESTIONS[0])

    # ──────────────────────────────────────────

    def _acc_bucket(self) -> int:
        if len(self.recent_results) < 3:
            return 1
        acc = np.mean(self.recent_results[-6:])
        return 0 if acc < 0.35 else (1 if acc < 0.65 else 2)

    def _knowledge_bucket(self, topic: str) -> int:
        k = self.student.knowledge[topic]
        return 0 if k < 0.35 else (1 if k < 0.65 else 2)

    def _fatigue_bucket(self) -> int:
        f = self.student.fatigue
        return 0 if f < 0.20 else (1 if f < 0.40 else 2)

    def _progress_bucket(self) -> int:
        p = self.step_count / self.max_steps
        return 0 if p < 0.33 else (1 if p < 0.66 else 2)

    def _get_state(self, question: dict) -> tuple:
        return (
            TOPICS.index(question["topic"]),
            question["difficulty"],
            self._acc_bucket(),
            self._knowledge_bucket(question["topic"]),
            self._fatigue_bucket(),
            self._progress_bucket(),
            1 if question["id"] in self.asked_ids else 0,
        )

    def get_state_array(self, question: dict) -> np.ndarray:
        """Numeric array of the state (for DQN input)."""
        s = self._get_state(question)
        return np.array(s, dtype=np.float32)

    # ──────────────────────────────────────────

    def step(self, action: int):
        question = QUESTIONS[action]

        # Repeat penalty
        is_repeat = action in self.asked_ids
        repeat_penalty = -0.8 if is_repeat else 0.0
        self.asked_ids.add(action)

        correct = self.student.answer(question)
        self.recent_results.append(1 if correct else 0)
        self.step_count += 1

        # ── Reward shaping ──────────────────────
        acc = self._acc_bucket()
        kb  = self._knowledge_bucket(question["topic"])
        d   = question["difficulty"]
        fat = self._fatigue_bucket()

        if correct:
            # Reward matching difficulty to knowledge level
            match_bonus = 0.5 if d == kb else 0.0
            base = [0.5, 0.9, 1.4][d] + match_bonus
            # Small bonus for pushing a good student harder
            if acc == 2 and d == 2:
                base += 0.3
        else:
            base = [-0.15, -0.45, -0.90][d]
            # Extra penalty for giving hard questions to struggling/tired student
            if acc == 0 and d == 2:
                base -= 0.6
            if fat == 2 and d == 2:
                base -= 0.3

        reward     = base + repeat_penalty
        done       = self.step_count >= self.max_steps
        next_state = self._get_state(question)

        info = {
            "question_id":      question["id"],
            "question_text":    question["text"],
            "topic":            question["topic"],
            "difficulty":       DIFFICULTY_NAMES[d],
            "correct":          correct,
            "reward":           reward,
            "student_knowledge":dict(self.student.knowledge),
            "fatigue":          self.student.fatigue,
            "confidence":       self.student.confidence,
            "is_repeat":        is_repeat,
        }
        return next_state, reward, done, info

    # ──────────────────────────────────────────

    @property
    def state_dim(self):
        return 7

    @property
    def n_actions(self):
        return N_ACTIONS


# ═══════════════════════════════════════════════════════════════
# RL AGENTS  (Q-Learning, UCB Bandit, DQN)
# ═══════════════════════════════════════════════════════════════
# agents.py — inlined
# TOPICS, N_ACTIONS already in scope

# agents.py — inlined
# TOPICS, N_ACTIONS already in scope

# agents.py — inlined
# TOPICS, N_ACTIONS already in scope

import numpy as np
import pickle
from collections import deque



# ─────────────────────────────────────────────────────────────
# 1. Q-LEARNING AGENT  (tabular)
# ─────────────────────────────────────────────────────────────

## Cell 4 — RL Agents: Q-Learning, UCB Bandit, DQN

### Agent 1: Tabular Q-Learning (Value-Based)
Maintains a Q-table mapping (state, action) → expected cumulative reward.

$$Q(s,a) \leftarrow Q(s,a) + \alpha \left[ r + \gamma \max_{a'} Q(s',a') - Q(s,a) \right]$$

| Hyperparameter | Value | Meaning |
|----------------|-------|--------|
| α (alpha) | 0.12 | Learning rate |
| γ (gamma) | 0.92 | Discount factor |
| ε (epsilon) | 1.0 → 0.05 | Exploration rate (decays over training) |

### Agent 2: UCB Contextual Bandit (Exploration Strategy)
Selects the best (topic, difficulty) zone for the current student context.

$$\text{UCB}(c,a) = \bar{Q}(c,a) + C\sqrt{\frac{\ln N_c}{N(c,a)}}$$

- **30 arms**: 10 topics × 3 difficulty levels
- **27 contexts**: acc_bucket × knowledge_bucket × fatigue_bucket
- **C = 1.4**: exploration constant

### Agent 3: DQN — Deep Q-Network (Value-Based, Neural)
A 3-layer neural network approximating the Q-function, implemented entirely in **pure NumPy** (no PyTorch or TensorFlow required).

```
Input (7) → Dense(64, ReLU) → Dense(64, ReLU) → Output(500 Q-values)
```

Key techniques:
- **Experience replay**: 10,000-transition replay buffer, random mini-batches of 64
- **Target network**: separate network synced every 100 steps for stable learning
- **Adam optimiser**: implemented from scratch with momentum and RMSProp correction

In [ ]:
class QLearningAgent:
    """
    Classic tabular Q-Learning.

    Bellman update:
        Q(s,a) ← Q(s,a) + α [ r + γ max_a' Q(s',a') - Q(s,a) ]

    State is a 7-tuple → dict key.
    """

    def __init__(
        self,
        n_actions:      int   = N_ACTIONS,
        alpha:          float = 0.12,
        gamma:          float = 0.92,
        epsilon:        float = 1.0,
        epsilon_min:    float = 0.05,
        epsilon_decay:  float = 0.997,
    ):
        self.n_actions     = n_actions
        self.alpha         = alpha
        self.gamma         = gamma
        self.epsilon       = epsilon
        self.epsilon_min   = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.q_table: dict = {}
        self.epsilons: list = []
        self.updates:  int  = 0

    def _q(self, state: tuple) -> np.ndarray:
        if state not in self.q_table:
            self.q_table[state] = np.zeros(self.n_actions, dtype=np.float32)
        return self.q_table[state]

    def select_action(self, state: tuple, greedy: bool = False) -> int:
        if not greedy and np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)
        q = self._q(state)
        ties = np.flatnonzero(q == q.max())
        return int(np.random.choice(ties))

    def update(self, state, action, reward, next_state, done):
        td_target = reward if done else reward + self.gamma * self._q(next_state).max()
        td_error  = td_target - self._q(state)[action]
        self.q_table[state][action] += self.alpha * td_error
        self.updates += 1

    def decay_epsilon(self):
        self.epsilons.append(self.epsilon)
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def save(self, path):
        with open(path, "wb") as f:
            pickle.dump(self.__dict__, f)

    def load(self, path):
        with open(path, "rb") as f:
            self.__dict__.update(pickle.load(f))


# ─────────────────────────────────────────────────────────────
# 2. UCB CONTEXTUAL BANDIT
# ─────────────────────────────────────────────────────────────

class UCBContextualBandit:
    """
    Contextual Bandit over (topic, difficulty) arms.

    Context = (acc_bucket, knowledge_bucket, fatigue_bucket) — 27 contexts.
    Arms    = (topic_idx, difficulty) — 10 × 3 = 30 arms.

    UCB score:
        UCB(c,a) = avg_reward(c,a) + C * sqrt( ln(N_c) / N(c,a) )
    """

    def __init__(self, C: float = 1.4):
        self.C    = C
        self.arms = [(t, d) for t in range(len(TOPICS)) for d in range(3)]  # 30 arms
        self.n_arms = len(self.arms)
        self.counts         = {}   # context → np.array(n_arms)
        self.rewards        = {}   # context → np.array(n_arms)
        self.context_counts = {}   # context → int

    def _init_context(self, ctx):
        if ctx not in self.counts:
            self.counts[ctx]  = np.zeros(self.n_arms, dtype=np.float64)
            self.rewards[ctx] = np.zeros(self.n_arms, dtype=np.float64)
            self.context_counts[ctx] = 0

    def select_arm(self, context: tuple) -> tuple:
        self._init_context(context)
        N_c   = self.context_counts[context]
        scores = np.zeros(self.n_arms)
        for i in range(self.n_arms):
            n = self.counts[context][i]
            scores[i] = (float('inf') if n == 0
                         else self.rewards[context][i]/n + self.C * np.sqrt(np.log(N_c + 1)/n))
        return self.arms[int(np.argmax(scores))]

    def update(self, context: tuple, arm: tuple, reward: float):
        self._init_context(context)
        i = self.arms.index(arm)
        self.counts[context][i]  += 1
        self.rewards[context][i] += reward
        self.context_counts[context] += 1

    def best_arm_per_context(self) -> dict:
        result = {}
        for ctx, counts in self.counts.items():
            totals = self.rewards.get(ctx, np.zeros(self.n_arms))
            avgs   = np.where(counts > 0, totals / np.maximum(counts, 1), -np.inf)
            best   = int(np.argmax(avgs))
            result[str(ctx)] = {"arm": self.arms[best], "avg_reward": float(avgs[best])}
        return result

    def get_arm_stats(self) -> dict:
        stats = {}
        for ctx, counts in self.counts.items():
            totals = self.rewards.get(ctx, np.zeros(self.n_arms))
            stats[str(ctx)] = {}
            for i, arm in enumerate(self.arms):
                n = counts[i]
                if n > 0:
                    stats[str(ctx)][str(arm)] = {
                        "count": int(n),
                        "avg_reward": float(totals[i] / n),
                    }
        return stats


# ─────────────────────────────────────────────────────────────
# 3. DQN AGENT  (Deep Q-Network — pure NumPy, no PyTorch needed)
# ─────────────────────────────────────────────────────────────

class DQNAgent:
    """
    A lightweight DQN implemented with NumPy only (no PyTorch/TF required).

    Architecture:  state(7) → Dense(64, ReLU) → Dense(64, ReLU) → Q-values(N_ACTIONS)

    Key techniques:
      • Experience replay   : random mini-batches from a replay buffer
      • Target network      : separate slowly-updated network for stable targets
      • ε-greedy exploration: decays over training
    """

    def __init__(
        self,
        state_dim:      int   = 7,
        n_actions:      int   = N_ACTIONS,
        hidden:         int   = 64,
        alpha:          float = 3e-4,
        gamma:          float = 0.92,
        epsilon:        float = 1.0,
        epsilon_min:    float = 0.05,
        epsilon_decay:  float = 0.997,
        buffer_size:    int   = 10_000,
        batch_size:     int   = 64,
        target_update:  int   = 100,   # steps between target network syncs
    ):
        self.state_dim    = state_dim
        self.n_actions    = n_actions
        self.alpha        = alpha
        self.gamma        = gamma
        self.epsilon      = epsilon
        self.epsilon_min  = epsilon_min
        self.epsilon_decay= epsilon_decay
        self.batch_size   = batch_size
        self.target_update= target_update
        self.steps        = 0
        self.epsilons: list = []

        # ── Network weights (online) ──────────────
        rng = np.random.default_rng(42)
        self.W1 = rng.normal(0, np.sqrt(2/state_dim),  (hidden, state_dim)).astype(np.float32)
        self.b1 = np.zeros(hidden,     dtype=np.float32)
        self.W2 = rng.normal(0, np.sqrt(2/hidden),     (hidden, hidden)).astype(np.float32)
        self.b2 = np.zeros(hidden,     dtype=np.float32)
        self.W3 = rng.normal(0, np.sqrt(2/hidden),     (n_actions, hidden)).astype(np.float32)
        self.b3 = np.zeros(n_actions,  dtype=np.float32)

        # ── Adam optimiser state ──────────────────
        self._adam_init()

        # ── Target network (copy of online) ──────
        self._sync_target()

        # ── Replay buffer ─────────────────────────
        self.buf_s  = np.zeros((buffer_size, state_dim), dtype=np.float32)
        self.buf_a  = np.zeros(buffer_size, dtype=np.int32)
        self.buf_r  = np.zeros(buffer_size, dtype=np.float32)
        self.buf_s2 = np.zeros((buffer_size, state_dim), dtype=np.float32)
        self.buf_d  = np.zeros(buffer_size, dtype=np.float32)
        self.buf_ptr= 0
        self.buf_len= 0
        self.buf_max= buffer_size

    # ── Forward pass ─────────────────────────────

    def _relu(self, x):
        return np.maximum(0, x)

    def _forward(self, x, use_target=False):
        """x shape: (batch, state_dim) or (state_dim,)"""
        flat = x.ndim == 1
        if flat:
            x = x[None]
        if use_target:
            h1 = self._relu(x @ self.tW1.T + self.tb1)
            h2 = self._relu(h1 @ self.tW2.T + self.tb2)
            out = h2 @ self.tW3.T + self.tb3
        else:
            h1 = self._relu(x @ self.W1.T + self.b1)
            h2 = self._relu(h1 @ self.W2.T + self.b2)
            out = h2 @ self.W3.T + self.b3
        return out[0] if flat else out

    # ── Adam optimiser ───────────────────────────

    def _adam_init(self):
        self.t = 0
        zeros = lambda shape: np.zeros(shape, dtype=np.float32)
        self.mW1=zeros(self.W1.shape); self.vW1=zeros(self.W1.shape)
        self.mb1=zeros(self.b1.shape); self.vb1=zeros(self.b1.shape)
        self.mW2=zeros(self.W2.shape); self.vW2=zeros(self.W2.shape)
        self.mb2=zeros(self.b2.shape); self.vb2=zeros(self.b2.shape)
        self.mW3=zeros(self.W3.shape); self.vW3=zeros(self.W3.shape)
        self.mb3=zeros(self.b3.shape); self.vb3=zeros(self.b3.shape)

    def _adam_step(self, param, grad, m, v, beta1=0.9, beta2=0.999, eps=1e-8):
        self.t += 1
        m[:] = beta1 * m + (1 - beta1) * grad
        v[:] = beta2 * v + (1 - beta2) * grad**2
        m_hat = m / (1 - beta1**self.t)
        v_hat = v / (1 - beta2**self.t)
        param -= self.alpha * m_hat / (np.sqrt(v_hat) + eps)

    def _sync_target(self):
        self.tW1 = self.W1.copy(); self.tb1 = self.b1.copy()
        self.tW2 = self.W2.copy(); self.tb2 = self.b2.copy()
        self.tW3 = self.W3.copy(); self.tb3 = self.b3.copy()

    # ── Experience replay ────────────────────────

    def store(self, s, a, r, s2, done):
        i = self.buf_ptr % self.buf_max
        self.buf_s[i]  = s
        self.buf_a[i]  = a
        self.buf_r[i]  = r
        self.buf_s2[i] = s2
        self.buf_d[i]  = float(done)
        self.buf_ptr   = (self.buf_ptr + 1) % self.buf_max
        self.buf_len   = min(self.buf_len + 1, self.buf_max)

    # ── Select action ────────────────────────────

    def select_action(self, state: np.ndarray, greedy: bool = False) -> int:
        if not greedy and np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)
        q = self._forward(state)
        return int(np.argmax(q))

    # ── Learn from replay buffer ─────────────────

    def learn(self):
        if self.buf_len < self.batch_size:
            return 0.0

        idx  = np.random.choice(self.buf_len, self.batch_size, replace=False)
        s    = self.buf_s[idx]
        a    = self.buf_a[idx]
        r    = self.buf_r[idx]
        s2   = self.buf_s2[idx]
        done = self.buf_d[idx]

        # Target Q values (using target network)
        q_next   = self._forward(s2, use_target=True)            # (batch, n_actions)
        td_target= r + self.gamma * q_next.max(axis=1) * (1 - done)

        # Online Q values
        q_pred   = self._forward(s)                              # (batch, n_actions)
        td_error = q_pred[np.arange(self.batch_size), a] - td_target   # (batch,)
        loss     = float(np.mean(td_error**2))

        # Backprop  (manual, layer by layer)
        # Output layer gradient
        dq         = np.zeros_like(q_pred)
        dq[np.arange(self.batch_size), a] = 2.0 * td_error / self.batch_size

        h1 = self._relu(s  @ self.W1.T + self.b1)
        h2 = self._relu(h1 @ self.W2.T + self.b2)

        dW3 = dq.T @ h2
        db3 = dq.sum(axis=0)

        dh2 = dq @ self.W3
        dh2[h2 <= 0] = 0   # ReLU gate

        dW2 = dh2.T @ h1
        db2 = dh2.sum(axis=0)

        dh1 = dh2 @ self.W2
        dh1[h1 <= 0] = 0

        dW1 = dh1.T @ s
        db1 = dh1.sum(axis=0)

        # Adam updates
        self._adam_step(self.W3, dW3, self.mW3, self.vW3)
        self._adam_step(self.b3, db3, self.mb3, self.vb3)
        self._adam_step(self.W2, dW2, self.mW2, self.vW2)
        self._adam_step(self.b2, db2, self.mb2, self.vb2)
        self._adam_step(self.W1, dW1, self.mW1, self.vW1)
        self._adam_step(self.b1, db1, self.mb1, self.vb1)

        self.steps += 1
        if self.steps % self.target_update == 0:
            self._sync_target()

        return loss

    def decay_epsilon(self):
        self.epsilons.append(self.epsilon)
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def save(self, path):
        arrays = {k: v for k, v in self.__dict__.items()
                  if isinstance(v, np.ndarray) or isinstance(v, (int, float, list))}
        with open(path, "wb") as f:
            pickle.dump(arrays, f)

    def load(self, path):
        with open(path, "rb") as f:
            self.__dict__.update(pickle.load(f))


# ═══════════════════════════════════════════════════════════════
# HYBRID AGENT + TRAINING
# ═══════════════════════════════════════════════════════════════
# train.py — inlined
# All dependencies (QUESTIONS, TOPICS, N_ACTIONS, TutoringEnvironment, etc.) in scope

# train.py — inlined
# All dependencies (QUESTIONS, TOPICS, N_ACTIONS, TutoringEnvironment, etc.) in scope

# train.py — inlined
# All dependencies (QUESTIONS, TOPICS, N_ACTIONS, TutoringEnvironment, etc.) in scope

import numpy as np
import json, os, time
from collections import deque



# ─────────────────────────────────────────────────────────────
# HYBRID AGENT
# ─────────────────────────────────────────────────────────────

## Cell 5 — Hybrid Agent & Training Loop

### `HybridTutoringAgent` — Three-Layer Decision Process

```
Step 1 → UCB Bandit:   given student context, pick best (topic, difficulty)
Step 2 → Q-Learning:   given zone, pick best specific question by Q-value
Step 3 → DQN ensemble: re-score with neural net, blend with Q-pick
         (DQN weight increases from 0% → 75% as training progresses)
```

### Training Protocol
- **500 episodes** × 40 steps each = 20,000 total interactions
- Each episode: fresh randomly-initialised student
- **Baseline**: random agent selecting questions uniformly at random
- Both agents use identical student seeds for fair comparison

In [ ]:
class HybridTutoringAgent:
    """
    Three-layer decision process:

      Layer 1 – UCB Bandit
        Looks at (acc_bucket, knowledge_bucket, fatigue_bucket) and
        recommends the best (topic, difficulty) zone to focus on.

      Layer 2 – Q-Learning
        Within the recommended zone, picks the best specific question
        according to its Q-table.

      Layer 3 – DQN
        Uses the full 7-dim state to re-score the Q-Learning candidate
        and the DQN's own top pick; final action is chosen by an
        ensemble weighted toward DQN as training progresses.
    """

    def __init__(self):
        self.q_agent = QLearningAgent()
        self.ucb     = UCBContextualBandit()
        self.dqn     = DQNAgent()
        self._step   = 0

    # ──────────────────────────────────────────────

    def select_action(
        self,
        state:      tuple,
        state_arr:  np.ndarray,
        env:        TutoringEnvironment,
        greedy:     bool = False,
    ) -> int:
        acc_b  = state[2]
        kb     = state[3]
        fat_b  = state[4]
        context = (acc_b, kb, fat_b)

        # Layer 1: UCB zone recommendation
        rec_topic, rec_diff = self.ucb.select_arm(context)

        # Layer 2: Q-agent within zone
        candidates = [
            q["id"] for q in QUESTIONS
            if TOPICS.index(q["topic"]) == rec_topic
            and q["difficulty"] == rec_diff
            and q["id"] not in env.asked_ids
        ]
        if candidates:
            q_vals   = self.q_agent._q(state)
            q_pick   = max(candidates, key=lambda a: q_vals[a])
        else:
            q_pick   = self.q_agent.select_action(state, greedy=greedy)

        # Layer 3: DQN picks its own best action
        dqn_q    = self.dqn._forward(state_arr)
        # mask already-asked
        for asked in env.asked_ids:
            dqn_q[asked] -= 1e6
        dqn_pick = int(np.argmax(dqn_q))

        # Ensemble: blend Q-pick and DQN-pick
        # Early in training: trust Q-learning more (DQN hasn't learned yet)
        # Later: trust DQN more (has seen more diverse states)
        dqn_weight = min(0.75, self._step / 30_000)
        if np.random.random() < dqn_weight:
            return dqn_pick
        return q_pick

    # ──────────────────────────────────────────────

    def update(
        self,
        state,  state_arr,
        action, reward,
        next_state, next_state_arr,
        done,
    ):
        context  = (state[2], state[3], state[4])
        topic_i  = TOPICS.index(QUESTIONS[action]["topic"])
        diff_i   = QUESTIONS[action]["difficulty"]

        self.q_agent.update(state, action, reward, next_state, done)
        self.ucb.update(context, (topic_i, diff_i), reward)
        self.dqn.store(state_arr, action, reward, next_state_arr, done)
        self.dqn.learn()

        self._step += 1

    def decay_epsilon(self):
        self.q_agent.decay_epsilon()
        self.dqn.decay_epsilon()

    def save(self, save_dir: str):
        os.makedirs(save_dir, exist_ok=True)
        self.q_agent.save(os.path.join(save_dir, "q_agent.pkl"))
        self.dqn.save(os.path.join(save_dir, "dqn_agent.pkl"))
        with open(os.path.join(save_dir, "ucb_stats.json"), "w") as f:
            json.dump(self.ucb.get_arm_stats(), f, indent=2)

    def load(self, save_dir: str):
        self.q_agent.load(os.path.join(save_dir, "q_agent.pkl"))
        self.dqn.load(os.path.join(save_dir, "dqn_agent.pkl"))


# ─────────────────────────────────────────────────────────────
# TRAINING LOOP
# ─────────────────────────────────────────────────────────────

def run_training(
    n_episodes:    int  = 1000,
    max_steps:     int  = 40,
    save_dir:      str  = "results",
    verbose_every: int  = 100,
):
    os.makedirs(save_dir, exist_ok=True)
    agent = HybridTutoringAgent()

    ep_rewards    = []
    ep_accuracies = []
    ep_knowledge  = []
    ep_epsilons   = []
    dqn_losses    = []

    reward_window   = deque(maxlen=30)
    accuracy_window = deque(maxlen=30)

    t0 = time.time()
    print(f"Training: {n_episodes} episodes × {max_steps} steps/episode")
    print(f"Question bank: {N_ACTIONS} questions  |  Topics: {len(TOPICS)}")
    print("─" * 65)

    for ep in range(n_episodes):
        student   = StudentSimulator(seed=ep)
        env       = TutoringEnvironment(student=student, max_steps=max_steps)
        state     = env.reset(student=student)
        state_arr = env.get_state_array(QUESTIONS[0])

        total_r = 0.0
        ep_loss = []

        for _ in range(max_steps):
            action = agent.select_action(state, state_arr, env)
            next_state, reward, done, info = env.step(action)

            next_q   = QUESTIONS[action]
            next_arr = env.get_state_array(next_q)

            agent.update(state, state_arr, action, reward, next_state, next_arr, done)

            if agent.dqn.buf_len >= agent.dqn.batch_size:
                loss = agent.dqn.learn()
                ep_loss.append(loss)

            state     = next_state
            state_arr = next_arr
            total_r  += reward
            if done:
                break

        agent.decay_epsilon()

        ep_rewards.append(total_r)
        ep_accuracies.append(student.accuracy())
        ep_knowledge.append(student.overall_knowledge())
        ep_epsilons.append(agent.q_agent.epsilon)
        dqn_losses.append(float(np.mean(ep_loss)) if ep_loss else 0.0)

        reward_window.append(total_r)
        accuracy_window.append(student.accuracy())

        if (ep + 1) % verbose_every == 0 or ep == 0:
            elapsed = time.time() - t0
            print(
                f"  Ep {ep+1:5d}/{n_episodes} | "
                f"Reward: {np.mean(reward_window):+6.2f} | "
                f"Acc: {np.mean(accuracy_window):.1%} | "
                f"Know: {ep_knowledge[-1]:.2f} | "
                f"ε: {agent.q_agent.epsilon:.3f} | "
                f"DQN loss: {dqn_losses[-1]:.4f} | "
                f"{elapsed:.0f}s"
            )

    print("─" * 65)
    print(f"✅ Training complete in {time.time()-t0:.1f}s")

    results = {
        "episode_rewards":    ep_rewards,
        "episode_accuracies": ep_accuracies,
        "episode_knowledge":  ep_knowledge,
        "episode_epsilons":   ep_epsilons,
        "dqn_losses":         dqn_losses,
        "n_episodes":         n_episodes,
        "max_steps":          max_steps,
        "n_questions":        N_ACTIONS,
        "n_topics":           len(TOPICS),
    }
    with open(os.path.join(save_dir, "training_results.json"), "w") as f:
        json.dump(results, f, indent=2)

    agent.save(save_dir)
    print(f"Results + model saved to '{save_dir}/'")
    return agent, results


# ─────────────────────────────────────────────────────────────
# BASELINE  (random agent)
# ─────────────────────────────────────────────────────────────

def run_baseline(
    n_episodes: int = 1000,
    max_steps:  int = 40,
    save_dir:   str = "results",
):
    os.makedirs(save_dir, exist_ok=True)
    rewards, accuracies, knowledge = [], [], []

    for ep in range(n_episodes):
        student = StudentSimulator(seed=ep)
        env     = TutoringEnvironment(student=student, max_steps=max_steps)
        env.reset(student=student)
        total_r = 0.0
        for _ in range(max_steps):
            _, r, done, _ = env.step(np.random.randint(N_ACTIONS))
            total_r += r
            if done:
                break
        rewards.append(total_r)
        accuracies.append(student.accuracy())
        knowledge.append(student.overall_knowledge())

    results = {"episode_rewards": rewards, "episode_accuracies": accuracies,
               "episode_knowledge": knowledge}
    with open(os.path.join(save_dir, "baseline_results.json"), "w") as f:
        json.dump(results, f, indent=2)
    return results


# ─────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("\n── Baseline ─────────────────────────────────────────────")
    bl = run_baseline(n_episodes=1000)
    print(f"Baseline avg accuracy: {np.mean(bl['episode_accuracies']):.1%}")

    print("\n── RL Training ──────────────────────────────────────────")
    agent, rl = run_training(n_episodes=1000)

    bl_acc = np.mean(bl["episode_accuracies"][-200:])
    rl_acc = np.mean(rl["episode_accuracies"][-200:])
    print(f"\nFinal 200-ep accuracy  — Baseline: {bl_acc:.1%}  |  RL: {rl_acc:.1%}")
    print(f"Improvement: +{(rl_acc-bl_acc)*100:.1f} pp")


# ═══════════════════════════════════════════════════════════════
# TRAIN THE AGENT  (500 episodes — takes ~2 minutes)
# ═══════════════════════════════════════════════════════════════

## Cell 6 — Train the Agent (~2 minutes)

Runs 500 training episodes and saves results to `/content/results/`.
A random baseline is run first for comparison.

In [ ]:
print("Training baseline...")
baseline = run_baseline(n_episodes=500, save_dir='/content/results')
print(f"Baseline accuracy: {np.mean(baseline['episode_accuracies']):.1%}")

print("Training RL agent (500 episodes)...")
agent, results = run_training(n_episodes=500, save_dir='/content/results', verbose_every=100)

bl_acc = np.mean(baseline['episode_accuracies'][-100:])
rl_acc = np.mean(results['episode_accuracies'][-100:])
print(f"\nBaseline: {bl_acc:.1%}  |  RL Agent: {rl_acc:.1%}  |  Improvement: +{(rl_acc-bl_acc)*100:.1f}pp")

## Cell 7 — Results & Visualisations

Generates 4 figures from the training results:
- **Fig 1**: Learning curves (accuracy + reward) — RL vs baseline
- **Fig 2**: Early vs late training bar chart
- **Fig 3**: DQN loss + epsilon decay
- **Fig 4**: Summary statistics table

All figures are also saved to `/content/results/` for download.

In [ ]:
%matplotlib inline
import matplotlib
# matplotlib.use('Agg') if 'google.colab' not in str(get_ipython()) else None
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np, json

plt.rcParams['figure.dpi'] = 130
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size'] = 11

with open('/content/results/training_results.json') as f: rl = json.load(f)
with open('/content/results/baseline_results.json') as f: bl = json.load(f)

def smooth(vals, w=25):
    arr = np.array(vals, dtype=float)
    out = np.convolve(arr, np.ones(w)/w, mode='valid')
    return np.concatenate([np.full(w-1, out[0]), out])

BLUE   = '#1a73e8'
RED    = '#e53935'
GREEN  = '#2e7d32'
PURPLE = '#6c3fc5'
ORANGE = '#f57c00'

eps = np.arange(len(rl['episode_accuracies']))

# ── Figure 1: Learning Curves ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Figure 1 — Learning Curves: RL Agent vs Random Baseline',
             fontsize=14, fontweight='bold', y=1.02)

for ax, key, ylabel, pct in zip(
    axes,
    ['episode_accuracies', 'episode_rewards'],
    ['Student Answer Accuracy', 'Episode Reward'],
    [True, False]
):
    rl_s = smooth(rl[key]); bl_s = smooth(bl[key])
    ax.plot(eps, bl_s, color=RED,  label='Random Baseline', linewidth=2, alpha=0.85)
    ax.plot(eps, rl_s, color=BLUE, label='RL Agent',        linewidth=2.5)
    ax.fill_between(eps, bl_s, rl_s,
                    where=rl_s >= bl_s, alpha=0.12, color=BLUE, label='RL advantage')
    ax.set_xlabel('Episode'); ax.set_ylabel(ylabel)
    ax.legend(framealpha=0.9); ax.grid(True, alpha=0.25)
    if pct:
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f'{y:.0%}'))
        # Annotate final values
        ax.annotate(f'{rl_s[-1]:.0%}', xy=(len(eps)-1, rl_s[-1]),
                    xytext=(-60, 10), textcoords='offset points',
                    color=BLUE, fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color=BLUE, lw=1.5))
        ax.annotate(f'{bl_s[-1]:.0%}', xy=(len(eps)-1, bl_s[-1]),
                    xytext=(-60, -20), textcoords='offset points',
                    color=RED, fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color=RED, lw=1.5))

plt.tight_layout()
plt.savefig('/content/results/fig1_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig1_learning_curves.png')

# ── Figure 2: Early vs Late Training ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Figure 2 — Agent Improvement: Early vs Late Training',
             fontsize=14, fontweight='bold', y=1.02)

n = len(rl['episode_accuracies'])
for ax, key, title, pct in zip(
    axes,
    ['episode_accuracies', 'episode_rewards'],
    ['Student Accuracy', 'Episode Reward'],
    [True, False]
):
    groups = {
        f'Early RL\n(ep 1-{min(100,n//5)})':  np.array(rl[key][:min(100,n//5)]),
        f'Late RL\n(ep {n-100}-{n})':          np.array(rl[key][-100:]),
        'Random\nBaseline':                     np.array(bl[key]),
    }
    colors = [BLUE, GREEN, RED]
    means = [g.mean() for g in groups.values()]
    stds  = [g.std()  for g in groups.values()]
    bars  = ax.bar(list(groups.keys()), means, color=colors,
                   width=0.5, alpha=0.85, yerr=stds, capsize=7,
                   error_kw=dict(elinewidth=2, capthick=2))
    for bar, m, s in zip(bars, means, stds):
        lbl = f'{m:.1%}' if pct else f'{m:.2f}'
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + s + (0.005 if pct else 0.3),
                lbl, ha='center', va='bottom', fontweight='bold', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.set_ylabel(title)
    ax.grid(axis='y', alpha=0.25)
    if pct:
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f'{y:.0%}'))

plt.tight_layout()
plt.savefig('/content/results/fig2_early_vs_late.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig2_early_vs_late.png')

# ── Figure 3: DQN Loss + Epsilon Decay ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Figure 3 — DQN Training Dynamics',
             fontsize=14, fontweight='bold', y=1.02)

# DQN Loss
losses = [l for l in rl.get('dqn_losses', []) if l > 0]
if losses:
    sl = smooth(losses, w=min(20, len(losses)//2))
    axes[0].plot(sl, color=PURPLE, linewidth=2)
    axes[0].fill_between(range(len(sl)), sl, alpha=0.15, color=PURPLE)
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('DQN Training Loss  (↓ = learning)', fontsize=12)
axes[0].grid(True, alpha=0.25)
if losses:
    axes[0].annotate('Converging', xy=(len(sl)-1, sl[-1]),
                     xytext=(-80, 20), textcoords='offset points',
                     color=PURPLE, fontweight='bold',
                     arrowprops=dict(arrowstyle='->', color=PURPLE))

# Epsilon decay
epsilons = rl.get('episode_epsilons', [])
if epsilons:
    axes[1].plot(epsilons, color=ORANGE, linewidth=2.5)
    axes[1].fill_between(range(len(epsilons)), epsilons, alpha=0.15, color=ORANGE)
    # Annotate phases
    mid = len(epsilons) // 2
    axes[1].annotate('Exploring', xy=(10, epsilons[10]),
                     xytext=(30, epsilons[10]+0.05),
                     color='gray', fontsize=10,
                     arrowprops=dict(arrowstyle='->', color='gray'))
    axes[1].annotate('Exploiting', xy=(len(epsilons)-20, epsilons[-20]),
                     xytext=(len(epsilons)-120, epsilons[-20]+0.12),
                     color='gray', fontsize=10,
                     arrowprops=dict(arrowstyle='->', color='gray'))
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Epsilon (ε)')
axes[1].set_title('Exploration Rate Decay  (ε-greedy)', fontsize=12)
axes[1].set_ylim(0, 1.05)
axes[1].grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig('/content/results/fig3_dqn_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig3_dqn_dynamics.png')

# ── Figure 4: Summary Stats Table ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.axis('off')
fig.suptitle('Figure 4 — Training Summary Statistics',
             fontsize=14, fontweight='bold')

bl_acc_f = np.mean(bl['episode_accuracies'][-100:])
rl_acc_f = np.mean(rl['episode_accuracies'][-100:])
bl_rew_f = np.mean(bl['episode_rewards'][-100:])
rl_rew_f = np.mean(rl['episode_rewards'][-100:])

rows = [
    ['Metric', 'Random Baseline', 'RL Agent', 'Improvement'],
    ['Final Accuracy (last 100 ep)', f"{bl_acc_f:.1%}", f"{rl_acc_f:.1%}",
     f"+{(rl_acc_f-bl_acc_f)*100:.1f} pp"],
    ['Final Avg Reward (last 100 ep)', f"{bl_rew_f:.2f}", f"{rl_rew_f:.2f}",
     f"+{rl_rew_f-bl_rew_f:.2f}"],
    ['Episodes Trained', '—', str(rl['n_episodes']), '—'],
    ['Questions Available', '—', str(rl['n_questions']), '—'],
    ['Topics', '—', str(rl['n_topics']), '—'],
    ['RL Methods', '—', 'Q-Learning + UCB + DQN', '—'],
]

col_colors = [['#e8eaf6']*4] + [['white' if i%2==0 else '#f8f9ff']*4 for i in range(len(rows)-1)]
table = ax.table(cellText=rows[1:], colLabels=rows[0],
                 cellLoc='center', loc='center',
                 cellColours=col_colors[1:],
                 colColours=['#3f51b5']*4)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.2)
for j in range(4):
    table[0,j].set_text_props(color='white', fontweight='bold')
# Colour the improvement column green
for i in range(1, len(rows)):
    table[i,3].set_text_props(color='#2e7d32', fontweight='bold')

plt.tight_layout()
plt.savefig('/content/results/fig4_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig4_summary.png')

print('✅ All 4 figures generated and saved to /content/results/')
print('   You can download them from the Files panel on the left.')


## Cell 8 — Launch Interactive Tutor (Flask Web App)

Starts a Flask server with the multiple-choice quiz interface.
**Click the URL printed below to open the tutor in your browser.**

### Features
- A/B/C/D multiple choice with instant feedback
- Correct answer revealed after each question
- Live accuracy, score, and progress bar
- Knowledge level bars per topic
- Session summary with final knowledge breakdown

### How the RL agent adapts in real time
Every answer you submit updates the Q-table and DQN weights. The agent continuously adjusts which topics and difficulty levels to focus on based on your performance within the session.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# FLASK WEB APP  (Multiple Choice Interface)
# ═══════════════════════════════════════════════════════════════
HTML = "<!DOCTYPE html>\n<html>\n<head><title>Adaptive Science Tutor</title>\n<meta name='viewport' content='width=device-width, initial-scale=1'>\n<style>\n*{box-sizing:border-box;margin:0;padding:0}\nbody{font-family:'Segoe UI',Arial,sans-serif;background:#f0f4ff;min-height:100vh;padding:20px}\n.wrap{max-width:780px;margin:0 auto}\nh1{color:#1a73e8;font-size:1.9rem;margin-bottom:4px}\n.sub{color:#888;font-size:.9rem;margin-bottom:20px}\n.stats{display:flex;gap:12px;flex-wrap:wrap;margin-bottom:18px}\n.stat{background:white;border-radius:10px;padding:12px 18px;text-align:center;box-shadow:0 2px 8px rgba(0,0,0,.08);min-width:90px;flex:1}\n.sv{font-size:1.6rem;font-weight:700;color:#1a73e8}.sl{font-size:.7rem;color:#999;text-transform:uppercase;letter-spacing:.5px}\n.card{background:white;border-left:5px solid #1a73e8;border-radius:12px;padding:22px;margin-bottom:16px;box-shadow:0 2px 10px rgba(0,0,0,.08)}\n.meta{color:#888;font-size:.85rem;margin-bottom:10px;display:flex;gap:10px;align-items:center}\n.badge{padding:3px 10px;border-radius:20px;font-size:.8rem;font-weight:600}\n.easy{background:#e8f5e9;color:#2e7d32}.medium{background:#fff8e1;color:#f57f17}.hard{background:#fce4ec;color:#880e4f}\n.q{font-size:1.15rem;font-weight:600;color:#1a1a2e;margin-bottom:18px;line-height:1.4}\n.choices{display:grid;grid-template-columns:1fr 1fr;gap:10px;margin-bottom:6px}\n.choice{padding:12px 16px;border:2px solid #e0e0e0;border-radius:8px;background:white;cursor:pointer;font-size:.95rem;text-align:left;transition:all .15s;line-height:1.3;width:100%}\n.choice:hover:not([disabled]){border-color:#1a73e8;background:#f0f4ff}\n.choice.correct{border-color:#2e7d32 !important;background:#e8f5e9 !important;color:#1b5e20;font-weight:600}\n.choice.wrong{border-color:#c62828 !important;background:#ffebee !important;color:#b71c1c}\n.feedback{padding:12px 16px;border-radius:8px;font-weight:600;margin:10px 0;font-size:1rem}\n.fb-ok{background:#e8f5e9;color:#2e7d32;border:1px solid #a5d6a7}\n.fb-err{background:#ffebee;color:#c62828;border:1px solid #ef9a9a}\n.next-btn{background:#1a73e8;color:white;border:none;border-radius:8px;padding:11px 28px;font-size:1rem;font-weight:600;cursor:pointer;margin-top:6px}\n.start-btn{background:#1a73e8;color:white;border:none;border-radius:12px;padding:16px 36px;font-size:1.1rem;font-weight:600;cursor:pointer;margin-top:8px}\n.kbar{display:flex;align-items:center;gap:8px;margin:3px 0;font-size:.82rem}\n.klbl{width:145px;flex-shrink:0}.bw{flex:1;background:#eee;border-radius:6px;height:9px}\n.bf{height:100%;border-radius:6px}.kpct{width:34px;text-align:right;font-weight:600}\n.prog{background:#e8eaf6;border-radius:8px;height:8px;margin-bottom:18px;overflow:hidden}\n.prog-fill{height:100%;background:linear-gradient(90deg,#1a73e8,#6c3fc5);border-radius:8px;transition:width .4s}\ndetails summary{cursor:pointer;color:#1a73e8;margin:8px 0;font-size:.9rem}\n@media(max-width:500px){.choices{grid-template-columns:1fr}}\n</style>\n</head>\n<body><div class='wrap'>\n<h1>Adaptive Science Tutor</h1>\n<p class='sub'>Q-Learning + UCB Bandit + DQN &nbsp;|&nbsp; 500 questions &nbsp;|&nbsp; 10 topics</p>\n<div id='app'></div>\n</div>\n<script>\nvar S = {}, answered = false;\n\nfunction api(url, data, cb) {\n  fetch(url, {method:'POST', headers:{'Content-Type':'application/json'}, body:JSON.stringify(data)})\n    .then(function(r){ return r.json(); })\n    .then(cb)\n    .catch(function(e){ console.error(e); });\n}\n\nfunction startSession() {\n  api('/start', {}, function(data){ S = data; answered = false; render(); });\n}\n\nfunction submitAnswer(btn, chosen) {\n  if (answered) return;\n  answered = true;\n  var btns = document.querySelectorAll('.choice');\n  btns.forEach(function(b) {\n    b.disabled = true;\n    if (b.dataset.val === S.question.correct) b.classList.add('correct');\n    if (b.dataset.val === chosen && chosen !== S.question.correct) b.classList.add('wrong');\n  });\n  var fb = document.getElementById('feedback');\n  if (chosen === S.question.correct) {\n    fb.innerHTML = '<div class=\"feedback fb-ok\">Correct! Well done.</div>';\n  } else {\n    fb.innerHTML = '<div class=\"feedback fb-err\">Not quite. Correct answer: <b>' + S.question.correct + '</b></div>';\n  }\n  document.getElementById('next-btn').style.display = 'block';\n  api('/answer', {correct: chosen === S.question.correct}, function(data){ S = data; });\n}\n\nfunction nextQuestion() {\n  if (S.done) { render(); return; }\n  answered = false;\n  render();\n}\n\nfunction kbars(know) {\n  return Object.keys(know).map(function(t) {\n    var v = know[t], p = Math.round(v*100);\n    var col = p < 35 ? '#e53935' : p < 65 ? '#f57c00' : '#2e7d32';\n    return '<div class=\"kbar\"><span class=\"klbl\">' + t.replace(/_/g,' ') + '</span>' +\n      '<div class=\"bw\"><div class=\"bf\" style=\"width:' + p + '%;background:' + col + '\"></div></div>' +\n      '<span class=\"kpct\" style=\"color:' + col + '\">' + p + '%</span></div>';\n  }).join('');\n}\n\nfunction render() {\n  var el = document.getElementById('app');\n  if (!S.started && !S.done) {\n    el.innerHTML = '<div style=\"text-align:center;padding:40px 0\">' +\n      '<p style=\"color:#555;margin-bottom:20px;font-size:1.05rem\">The AI tutor adapts questions to your knowledge level in real time.</p>' +\n      '<button class=\"start-btn\" onclick=\"startSession()\">Start Session</button></div>';\n    return;\n  }\n  if (S.done) {\n    el.innerHTML = '<div class=\"card\"><h2 style=\"color:#1a73e8;margin-bottom:16px\">Session Complete!</h2>' +\n      '<div class=\"stats\">' +\n      '<div class=\"stat\"><div class=\"sv\">' + S.accuracy + '</div><div class=\"sl\">Accuracy</div></div>' +\n      '<div class=\"stat\"><div class=\"sv\">' + S.correct + '/' + S.total + '</div><div class=\"sl\">Score</div></div>' +\n      '</div><h3 style=\"margin:16px 0 10px;color:#333\">Final Knowledge</h3>' +\n      kbars(S.knowledge) +\n      '<br><button class=\"start-btn\" onclick=\"startSession()\">New Session</button></div>';\n    return;\n  }\n  var q = S.question;\n  var pct = Math.round(S.total / 20 * 100);\n  var acc = S.total > 0 ? Math.round(S.correct / S.total * 100) + '%' : '--';\n  var letters = ['A','B','C','D'];\n  var choiceHTML = q.choices.map(function(c, i) {\n    return '<button class=\"choice\" data-val=\"' + c.replace(/\"/g,'&quot;') + '\" onclick=\"submitAnswer(this,this.dataset.val)\">' +\n      '<b>' + letters[i] + '.</b> ' + c + '</button>';\n  }).join('');\n  el.innerHTML =\n    '<div class=\"stats\">' +\n    '<div class=\"stat\"><div class=\"sv\">' + acc + '</div><div class=\"sl\">Accuracy</div></div>' +\n    '<div class=\"stat\"><div class=\"sv\">' + S.correct + '/' + S.total + '</div><div class=\"sl\">Score</div></div>' +\n    '<div class=\"stat\"><div class=\"sv\">Q' + (S.total+1) + '/20</div><div class=\"sl\">Progress</div></div>' +\n    '</div>' +\n    '<div class=\"prog\"><div class=\"prog-fill\" style=\"width:' + pct + '%\"></div></div>' +\n    '<div class=\"card\">' +\n    '<div class=\"meta\"><span>' + q.topic.replace(/_/g,' ') + '</span>' +\n    '<span class=\"badge ' + q.difficulty + '\">' + q.difficulty.toUpperCase() + '</span></div>' +\n    '<div class=\"q\">' + q.text + '</div>' +\n    '<div class=\"choices\">' + choiceHTML + '</div>' +\n    '<div id=\"feedback\"></div>' +\n    '<button id=\"next-btn\" class=\"next-btn\" onclick=\"nextQuestion()\" style=\"display:none\">Next Question &rarr;</button>' +\n    '</div>' +\n    '<details><summary>Knowledge levels</summary>' + kbars(S.knowledge) + '</details>';\n}\n\nrender();\n</script>\n</body></html>"

_app   = Flask(__name__)
_state = {}

@_app.route('/')
def index():
    return Response(HTML, mimetype='text/html')

@_app.route('/start', methods=['POST'])
def start():
    _state['student'] = StudentSimulator(seed=int(time.time()) % 1000)
    _state['env']     = TutoringEnvironment(student=_state['student'], max_steps=20)
    _state['env'].reset(student=_state['student'])
    _state['history'] = []
    _state['correct_count'] = 0
    q = _pick()
    return jsonify(started=True, done=False, question=q,
                   knowledge=_state['student'].knowledge, total=0, correct=0)

def _pick():
    env   = _state['env']
    state = env._get_state(QUESTIONS[0])
    qv    = agent.q_agent._q(state)
    cands = [q['id'] for q in QUESTIONS if q['id'] not in env.asked_ids]
    if not cands:
        return None
    action = max(cands, key=lambda a: qv[a])
    _state['action'] = action
    q = QUESTIONS[action]
    shuffled = shuffle_choices(q, seed=action + _state['env'].step_count)
    return {'id': action, 'text': q['text'], 'topic': q['topic'],
             'difficulty': DIFFICULTY_NAMES[q['difficulty']],
             'choices': shuffled['choices'], 'correct': q['correct']}

@_app.route('/answer', methods=['POST'])
def answer():
    data    = request.json
    correct = data.get('correct', False)
    action  = _state.get('action', 0)
    env     = _state['env']
    student = _state['student']
    history = _state['history']
    q_obj   = QUESTIONS[action]
    d       = q_obj['difficulty']

    student.total_answered += 1
    if correct:
        student.total_correct += 1
        _state['correct_count'] = _state.get('correct_count', 0) + 1
        student.knowledge[q_obj['topic']] = min(1.0, student.knowledge[q_obj['topic']] + [0.02, 0.035, 0.055][d])
        student.confidence = min(1.0, student.confidence + 0.02)
    else:
        student.knowledge[q_obj['topic']] = min(1.0, student.knowledge[q_obj['topic']] + [0.005, 0.009, 0.014][d])
        student.confidence = max(0.1, student.confidence - 0.03)
    student.fatigue = min(0.6, student.fatigue + student.fatigue_rate)
    env.recent_results.append(1 if correct else 0)
    env.asked_ids.add(action)
    env.step_count += 1
    history.append({'correct': correct})

    nc   = _state.get('correct_count', 0)
    done = env.step_count >= env.max_steps or len(history) >= 20
    if done:
        return jsonify(done=True,
                       accuracy=f'{nc / max(1, len(history)):.0%}',
                       total=len(history), correct=nc,
                       knowledge=student.knowledge)
    q = _pick()
    return jsonify(started=True, done=False, question=q,
                   knowledge=student.knowledge,
                   total=len(history), correct=nc)


# Global error handler for clean JSON error responses
@_app.errorhandler(500)
def server_error(e):
    return jsonify(error='Server error', message=str(e)), 500

@_app.errorhandler(404)
def not_found(e):
    return jsonify(error='Not found'), 404

# Kill anything on port 5000 and launch
subprocess.run(['fuser', '-k', '5000/tcp'], capture_output=True)
time.sleep(1)

t = threading.Thread(
    target=lambda: _app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False))
t.daemon = True
t.start()
time.sleep(3)

from google.colab.output import eval_js
_url = eval_js('google.colab.kernel.proxyPort(5000)')
print('=' * 60)
print('  ADAPTIVE SCIENCE TUTOR — Multiple Choice Edition')
print(f'  URL: {_url}')
print('=' * 60)